# CSDid Cross-Implementation Validation Report

**Date:** 2026-07-04

**This notebook is shipped PRE-EXECUTED with all outputs.** You can inspect every
test result without running anything. Re-execution is optional and requires a
clone of the Python `csdid` package (see Setup cell below).

**Datasets:** `mpdta.csv`, `sim_data.csv`, 6 simulated datasets (`tp2_const`-`tp10_const`),
`mpdta_extra.csv`, `mpdta_tvw.csv`, `factor_cov.csv`, `county_mortality_data.csv` (JEL)

**Tolerances:** |diff| < 1e-6 for ATT point estimates, |diff| < 1e-4 for standard errors

| Implementation | Source | Execution |
|---|---|---|
| **Python** `csdid` | [d2cml-ai/csdid](https://github.com/d2cml-ai/csdid) | Live in notebook |
| **R** `did` v2.5.0 | Callaway & Sant'Anna reference | Pre-computed CSVs |
| **Julia** `CSDid.jl` | Local implementation | Pre-computed CSVs |
| **Stata** `csdid_jl` | Julia wrapper for Stata | Pre-computed CSVs |

**Result: 65/65 tests PASS across all 4 implementations.**

## Setup

In [1]:
import sys, os, warnings, io, contextlib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

# ── Project root: directory containing this notebook ──
PROJECT = os.getcwd()  # open the notebook from the CSDid.jl root
TOL_ATT = 1e-6
TOL_SE  = 1e-4

# ═══════════════════════════════════════════════════════════════
# EXTERNAL DEPENDENCY: Python csdid package
# ═══════════════════════════════════════════════════════════════
# The notebook needs a clone of https://github.com/d2cml-ai/csdid
# to run the Python estimator and read R reference CSVs.
#
# If you have not cloned it yet:
#   git clone https://github.com/d2cml-ai/csdid  C:\csdid_python
#
# Then update the path below to point to your clone:
CSDID_ROOT = os.path.join(os.path.dirname(PROJECT), 'csdid_python')
# ═══════════════════════════════════════════════════════════════

PYREPO = os.path.join(CSDID_ROOT, 'csdid', 'test_csdid')

if not os.path.isdir(CSDID_ROOT):
    print("=" * 70)
    print("ERROR: Python csdid repo not found at:")
    print(f"  {CSDID_ROOT}")
    print()
    print("To re-execute this notebook, clone the repo first:")
    print("  git clone https://github.com/d2cml-ai/csdid  " + CSDID_ROOT)
    print()
    print("Or update CSDID_ROOT in this cell to your clone location.")
    print("NOTE: The notebook is shipped pre-executed — you can inspect")
    print("all results without re-running.")
    print("=" * 70)
    raise FileNotFoundError(f"csdid repo not found: {CSDID_ROOT}")

# ── Ensure we import from the local clone, not a system install ──
if CSDID_ROOT not in sys.path:
    sys.path.insert(0, CSDID_ROOT)
for _mod in [k for k in sys.modules if k == 'csdid' or k.startswith('csdid.')]:
    del sys.modules[_mod]

from csdid.att_gt import ATTgt
import csdid
print(f"csdid loaded from: {csdid.__file__}")

# ── Datasets ──
mpdta      = pd.read_csv(os.path.join(PROJECT, 'data', 'mpdta.csv'))
sim_data   = pd.read_csv(os.path.join(PYREPO, 'sim_data.csv'))
mpdta_extra= pd.read_csv(os.path.join(PROJECT, 'data', 'mpdta_extra.csv'))
mpdta_tvw  = pd.read_csv(os.path.join(PROJECT, 'data', 'mpdta_tvw.csv'))
factor_data= pd.read_csv(os.path.join(PROJECT, 'data', 'factor_cov.csv'))

def load_sim(name):
    return pd.read_csv(os.path.join(PROJECT, 'data', f'{name}.csv'))

# ── R references ──
r_attgt  = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'ref_attgt.csv'))
r_aggte  = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'ref_aggte.csv'))
r_sim    = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'sim', 'ref_sim.csv'))
r_gaps   = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'sim', 'ref_gaps.csv'))
r_fixwt  = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'ref_fixweights.csv'))
r_factor = pd.read_csv(os.path.join(PYREPO, 'r_ref', 'sim', 'ref_factor.csv'))

# ── Julia references ──
jl_attgt   = pd.read_csv(os.path.join(PROJECT, 'julia_results.csv'))
jl_aggte   = pd.read_csv(os.path.join(PROJECT, 'julia_aggte_results_full.csv'))
jl_jel     = pd.read_csv(os.path.join(PROJECT, 'julia_jel_results.csv'))
jl_jel_agg = pd.read_csv(os.path.join(PROJECT, 'julia_jel_aggte.csv'))

# ── Stata references ──
st_attgt = pd.read_csv(os.path.join(PROJECT, 'stata_results.csv'))
st_aggte = pd.read_csv(os.path.join(PROJECT, 'stata_aggte_results_full.csv'))

STATA_MAP = {
    'mpdta_nev_dr': 'dr_nev_nocov',
    'mpdta_nyt_dr': 'dr_nyt_nocov',
    'mpdta_nev_reg_cov': 'reg_nev_cov',
    'mpdta_nev_ipw': 'ipw_nev_nocov',
    'sim_nev_dr': 'sim_nev_dr',
}
_st_scenarios = set(st_attgt['scenario'].unique())

def get_st_scn(test_scn):
    """Get Stata scenario name, or None if unavailable."""
    if test_scn in STATA_MAP:
        return STATA_MAP[test_scn]
    if test_scn in _st_scenarios:
        return test_scn
    return None

# ── Result tracker ──
results_tracker = []

# ══════════════════════════════════════════════════════════════
# Reference lookup helpers
# ══════════════════════════════════════════════════════════════

def _ref_lookup(df, filters=None):
    """Extract {(group, t): (att, se)} from reference DataFrame."""
    sub = df
    if filters:
        for col, val in filters.items():
            sub = sub[sub[col] == val]
    if sub.empty:
        return {}
    return {(int(r['group']), int(r['t'])): (float(r['att']), float(r['se']))
            for _, r in sub.iterrows()}

def _ref_aggte_egt(df, filters, agg_type, type_col='type'):
    """Extract {egt: (att_egt, se_egt)} and (overall_att, overall_se)."""
    sub = df
    if filters:
        for col, val in filters.items():
            sub = sub[sub[col] == val]
    sub = sub[sub[type_col] == agg_type]
    if sub.empty:
        return {}, (np.nan, np.nan)
    ov_row = sub.iloc[0]
    overall = (float(ov_row['overall_att']), float(ov_row['overall_se']))
    egt_sub = sub[sub['egt'].notna()]
    egt_map = {float(r['egt']): (float(r['att_egt']), float(r['se_egt']))
               for _, r in egt_sub.iterrows()}
    return egt_map, overall

def _ref_aggte_overall(df, filters, type_col='type'):
    """Extract {agg_type: (overall_att, overall_se)}."""
    sub = df
    if filters:
        for col, val in filters.items():
            sub = sub[sub[col] == val]
    result = {}
    for tp in ['simple', 'dynamic', 'group', 'calendar']:
        tp_sub = sub[sub[type_col] == tp]
        if not tp_sub.empty:
            row = tp_sub.iloc[0]
            result[tp] = (float(row['overall_att']), float(row['overall_se']))
    return result

def _extract_py(res):
    """Extract {(g,t): (att, se)} from ATTgt result."""
    g = np.asarray(res.MP['group'], dtype=float)
    t = np.asarray(res.MP['t'], dtype=float)
    att = np.asarray(res.results['att'], dtype=float)
    se = np.asarray(res.results['se'], dtype=float)
    return {(int(gi), int(ti)): (ai, si) for gi, ti, ai, si in zip(g, t, att, se)}

# ══════════════════════════════════════════════════════════════
# compare_test: plain DataFrame comparison
# ══════════════════════════════════════════════════════════════

def compare_test(name, df, att_tol=TOL_ATT, se_tol=TOL_SE):
    """Add att_diff/se_diff/ok columns, print result, record PASS/FAIL.

    att_diff = max - min across all populated ATT columns (R, Py, Jl, St).
    se_diff  = max - min across all populated SE columns.
    This catches discrepancies in ANY implementation, not just Py vs R.
    """
    df = df.copy()

    def _max_diff(row, cols):
        vals = row[cols].dropna()
        return (vals.max() - vals.min()) if len(vals) >= 2 else np.nan

    att_cols = [c for c in ['R_att', 'Py_att', 'Jl_att', 'St_att'] if c in df.columns]
    se_cols  = [c for c in ['R_se',  'Py_se',  'Jl_se',  'St_se']  if c in df.columns]
    df['att_diff'] = df.apply(lambda r: _max_diff(r, att_cols), axis=1)
    df['se_diff']  = df.apply(lambda r: _max_diff(r, se_cols),  axis=1)

    def _ok(r):
        a = r.get('att_diff', np.nan)
        s = r.get('se_diff', np.nan)
        a_ok = pd.isna(a) or a < att_tol
        s_ok = pd.isna(s) or s < se_tol
        return 'PASS' if (a_ok and s_ok) else 'FAIL'

    df['ok'] = df.apply(_ok, axis=1)
    tag = 'FAIL' if (df['ok'] == 'FAIL').any() else 'PASS'
    results_tracker.append({'name': name, 'status': tag})
    n = len(df)
    print(f"{tag} {name} ({n} cells)")
    fcols = df.select_dtypes(include='float').columns
    df[fcols] = df[fcols].round(7)
    print(df.to_string(index=False))

# ══════════════════════════════════════════════════════════════
# Preparation helpers
# ══════════════════════════════════════════════════════════════

def prep_attgt(res, r_scn=None, jl_scn=None, st_scn=None,
               r_df=None, r_filt=None, jl_df=None):
    """Build comparison DataFrame for ATT(g,t) test."""
    py = _extract_py(res)
    r  = _ref_lookup(r_df if r_df is not None else r_attgt,
                     r_filt if r_filt else ({'scenario': r_scn} if r_scn else None))
    jl = _ref_lookup(jl_df if jl_df is not None else jl_attgt,
                     {'scenario': jl_scn}) if jl_scn else {}
    st = _ref_lookup(st_attgt, {'scenario': st_scn}) if st_scn else {}
    keys = sorted(set(py) | set(r) | set(jl) | set(st))
    rows = []
    for k in keys:
        py_a, py_s = py.get(k, (np.nan, np.nan))
        r_a, r_s = r.get(k, (np.nan, np.nan))
        jl_a, jl_s = jl.get(k, (np.nan, np.nan))
        st_a, st_s = st.get(k, (np.nan, np.nan))
        rows.append({
            'g': int(k[0]), 't': int(k[1]),
            'R_att': r_a, 'Py_att': py_a, 'Jl_att': jl_a, 'St_att': st_a,
            'R_se': r_s, 'Py_se': py_s, 'Jl_se': jl_s, 'St_se': st_s,
        })
    return pd.DataFrame(rows)

def prep_aggte_overall(res, r_scn, jl_scn, st_scn=None):
    """Build comparison DataFrame for overall aggregation (4 types)."""
    rows = []
    r_ov  = _ref_aggte_overall(r_aggte,  {'scenario': r_scn},  type_col='type')
    jl_ov = _ref_aggte_overall(jl_aggte, {'scenario': jl_scn}, type_col='agg_type')
    st_ov = _ref_aggte_overall(st_aggte, {'scenario': st_scn}, type_col='agg_type') if st_scn else {}
    for tp in ['simple', 'dynamic', 'group', 'calendar']:
        with contextlib.redirect_stdout(io.StringIO()):
            res.aggte(typec=tp, bstrap=False)
        py_att = float(np.ravel(res.atte['overall_att'])[0])
        py_se  = float(np.ravel(res.atte['overall_se'])[0])
        r_a,  r_s  = r_ov.get(tp,  (np.nan, np.nan))
        jl_a, jl_s = jl_ov.get(tp, (np.nan, np.nan))
        st_a, st_s = st_ov.get(tp, (np.nan, np.nan))
        rows.append({
            'g': tp, 't': '-',
            'R_att': r_a, 'Py_att': py_att, 'Jl_att': jl_a, 'St_att': st_a,
            'R_se': r_s, 'Py_se': py_se, 'Jl_se': jl_s, 'St_se': st_s,
        })
    return pd.DataFrame(rows)

def prep_aggte_egt(res, r_scn, jl_scn, agg_type, st_scn=None):
    """Build comparison DataFrame for per-egt aggregation + overall."""
    with contextlib.redirect_stdout(io.StringIO()):
        res.aggte(typec=agg_type, bstrap=False)
    py_egt_arr = np.asarray(res.atte['egt'], dtype=float)
    py_att_arr = np.asarray(res.atte['att_egt'], dtype=float)
    py_se_arr  = np.asarray(res.atte['se_egt'], dtype=float)
    py_ov_att  = float(np.ravel(res.atte['overall_att'])[0])
    py_ov_se   = float(np.ravel(res.atte['overall_se'])[0])
    py_map     = {float(e): (a, s) for e, a, s in zip(py_egt_arr, py_att_arr, py_se_arr)}

    r_map,  r_ov  = _ref_aggte_egt(r_aggte,  {'scenario': r_scn},  agg_type, type_col='type')
    jl_map, jl_ov = _ref_aggte_egt(jl_aggte, {'scenario': jl_scn}, agg_type, type_col='agg_type')
    if st_scn:
        st_map, st_ov = _ref_aggte_egt(st_aggte, {'scenario': st_scn}, agg_type, type_col='agg_type')
    else:
        st_map, st_ov = {}, (np.nan, np.nan)

    rows = [{'g': 'overall', 't': '-',
             'R_att': r_ov[0], 'Py_att': py_ov_att, 'Jl_att': jl_ov[0], 'St_att': st_ov[0],
             'R_se': r_ov[1], 'Py_se': py_ov_se, 'Jl_se': jl_ov[1], 'St_se': st_ov[1]}]

    all_egts = sorted(set(py_map) | set(r_map) | set(jl_map) | set(st_map))
    for e in all_egts:
        lbl = f'e={int(e)}' if abs(e) < 100 else f'g={int(e)}'
        rows.append({'g': lbl, 't': '-',
                     'R_att': r_map.get(e, (np.nan,np.nan))[0],
                     'Py_att': py_map.get(e, (np.nan,np.nan))[0],
                     'Jl_att': jl_map.get(e, (np.nan,np.nan))[0],
                     'St_att': st_map.get(e, (np.nan,np.nan))[0],
                     'R_se': r_map.get(e, (np.nan,np.nan))[1],
                     'Py_se': py_map.get(e, (np.nan,np.nan))[1],
                     'Jl_se': jl_map.get(e, (np.nan,np.nan))[1],
                     'St_se': st_map.get(e, (np.nan,np.nan))[1]})
    return pd.DataFrame(rows)

print(f"Setup complete. Python {sys.version.split()[0]}, NumPy {np.__version__}, Pandas {pd.__version__}")
print(f"Project: {PROJECT}")
print(f"Datasets loaded: mpdta ({len(mpdta)}), sim_data ({len(sim_data)}), mpdta_extra ({len(mpdta_extra)})")
print(f"R refs: attgt ({len(r_attgt)}), aggte ({len(r_aggte)}), sim ({len(r_sim)}), gaps ({len(r_gaps)}), fixwt ({len(r_fixwt)}), factor ({len(r_factor)})")
print(f"Julia refs: attgt ({len(jl_attgt)}), aggte ({len(jl_aggte)}), jel ({len(jl_jel)}), jel_agg ({len(jl_jel_agg)})")
print(f"Stata refs: attgt ({len(st_attgt)}), aggte ({len(st_aggte)}), scenarios: {len(_st_scenarios)}")

csdid loaded from: C:\Users\Usuario\csdid_python\csdid\__init__.py
Setup complete. Python 3.13.5, NumPy 2.2.6, Pandas 2.3.3
Project: C:\Users\Usuario\CSDid.jl
Datasets loaded: mpdta (2500), sim_data (15916), mpdta_extra (2500)
R refs: attgt (57), aggte (72), sim (660), gaps (59), fixwt (48), factor (9)
Julia refs: attgt (977), aggte (981), jel (144), jel_agg (87)
Stata refs: attgt (929), aggte (231), scenarios: 47


## A. test_r_parity: ATT(g,t) point estimates and SEs (5 tests)

### Test 1 — test_attgt_matches_r[mpdta_nev_dr]

`mpdta`, DR, nevertreated, no covariates

In [2]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, r_scn='mpdta_nev_dr', jl_scn='mpdta_nev_dr', st_scn='dr_nev_nocov')
compare_test('test_attgt_matches_r[mpdta_nev_dr]', df)

PASS test_attgt_matches_r[mpdta_nev_dr] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.010503 -0.010503 -0.010503 -0.010503 0.023251 0.023251 0.023251 0.023251 0.000000e+00      0.0 PASS
2004 2005 -0.070423 -0.070423 -0.070423 -0.070423 0.030985 0.030985 0.030985 0.030985 0.000000e+00      0.0 PASS
2004 2006 -0.137259 -0.137259 -0.137259 -0.137259 0.036436 0.036436 0.036436 0.036436 0.000000e+00      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.034359 0.034359 0.034359 0.034359 0.000000e+00      0.0 PASS
2006 2004  0.006520  0.006520  0.006520  0.006520 0.023327 0.023327 0.023327 0.023327 1.000000e-07      0.0 PASS
2006 2005 -0.002751 -0.002751 -0.002751 -0.002751 0.019559 0.019559 0.019559 0.019559 0.000000e+00      0.0 PASS
2006 2006 -0.004595 -0.004595 -0.004595 -0.004595 0.017755 0.017755 0.017755 0.017755 0.000000e+00      0.0 PASS
2006 2007 -0.041224 -0.041224 -0.041224 -0.04

### Test 2 — test_attgt_matches_r[mpdta_nyt_dr]

`mpdta`, DR, notyettreated, no covariates

In [3]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, r_scn='mpdta_nyt_dr', jl_scn='mpdta_nyt_dr', st_scn='dr_nyt_nocov')
compare_test('test_attgt_matches_r[mpdta_nyt_dr]', df)

PASS test_attgt_matches_r[mpdta_nyt_dr] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.019372 -0.019372 -0.019372 -0.019372 0.022310 0.022310 0.022310 0.022310 1.000000e-07      0.0 PASS
2004 2005 -0.078319 -0.078319 -0.078319 -0.078319 0.030390 0.030390 0.030390 0.030390 1.000000e-07      0.0 PASS
2004 2006 -0.136274 -0.136274 -0.136274 -0.136274 0.035403 0.035403 0.035403 0.035403 0.000000e+00      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.034359 0.034359 0.034359 0.034359 0.000000e+00      0.0 PASS
2006 2004 -0.002563 -0.002563 -0.002563 -0.002563 0.022530 0.022530 0.022530 0.022530 1.000000e-07      0.0 PASS
2006 2005 -0.001939 -0.001939 -0.001939 -0.001939 0.019042 0.019042 0.019042 0.019042 0.000000e+00      0.0 PASS
2006 2006  0.004661  0.004661  0.004661  0.004661 0.016336 0.016336 0.016336 0.016336 0.000000e+00      0.0 PASS
2006 2007 -0.041224 -0.041224 -0.041224 -0.04

### Test 3 — test_attgt_matches_r[mpdta_nev_reg_cov]

`mpdta`, REG, nevertreated, covariates: lpop

In [4]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~lpop', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, r_scn='mpdta_nev_reg_cov', jl_scn='mpdta_nev_reg_cov', st_scn='reg_nev_cov')
compare_test('test_attgt_matches_r[mpdta_nev_reg_cov]', df)

PASS test_attgt_matches_r[mpdta_nev_reg_cov] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.014911 -0.014911 -0.014911 -0.014911 0.022056 0.022056 0.022056 0.022056 0.000000e+00      0.0 PASS
2004 2005 -0.076996 -0.076996 -0.076996 -0.076996 0.028360 0.028360 0.028360 0.028360 0.000000e+00      0.0 PASS
2004 2006 -0.141080 -0.141080 -0.141080 -0.141080 0.034836 0.034836 0.034836 0.034836 0.000000e+00      0.0 PASS
2004 2007 -0.107544 -0.107544 -0.107544 -0.107544 0.032738 0.032738 0.032738 0.032738 0.000000e+00      0.0 PASS
2006 2004 -0.002066 -0.002066 -0.002066 -0.002066 0.022122 0.022122 0.022122 0.022122 1.000000e-07      0.0 PASS
2006 2005 -0.006968 -0.006968 -0.006968 -0.006968 0.018346 0.018346 0.018346 0.018346 0.000000e+00      0.0 PASS
2006 2006  0.000766  0.000766  0.000766  0.000766 0.019196 0.019196 0.019196 0.019196 0.000000e+00      0.0 PASS
2006 2007 -0.041536 -0.041536 -0.041536 

### Test 4 — test_attgt_matches_r[mpdta_nev_ipw]

`mpdta`, IPW, nevertreated, no covariates

In [5]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='ipw', bstrap=False)
df = prep_attgt(res, r_scn='mpdta_nev_ipw', jl_scn='mpdta_nev_ipw', st_scn='ipw_nev_nocov')
compare_test('test_attgt_matches_r[mpdta_nev_ipw]', df)

PASS test_attgt_matches_r[mpdta_nev_ipw] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.010503 -0.010503 -0.010503 -0.010503 0.023251 0.023251 0.023251 0.023251 0.000000e+00      0.0 PASS
2004 2005 -0.070423 -0.070423 -0.070423 -0.070423 0.030985 0.030985 0.030985 0.030985 0.000000e+00      0.0 PASS
2004 2006 -0.137259 -0.137259 -0.137259 -0.137259 0.036436 0.036436 0.036436 0.036436 0.000000e+00      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.034359 0.034359 0.034359 0.034359 0.000000e+00      0.0 PASS
2006 2004  0.006520  0.006520  0.006520  0.006520 0.023327 0.023327 0.023327 0.023327 1.000000e-07      0.0 PASS
2006 2005 -0.002751 -0.002751 -0.002751 -0.002751 0.019559 0.019559 0.019559 0.019559 0.000000e+00      0.0 PASS
2006 2006 -0.004595 -0.004595 -0.004595 -0.004595 0.017755 0.017755 0.017755 0.017755 0.000000e+00      0.0 PASS
2006 2007 -0.041224 -0.041224 -0.041224 -0.0

### Test 5 — test_attgt_matches_r[sim_nev_dr]

`sim_data`, DR, nevertreated, covariates: X

In [6]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, r_scn='sim_nev_dr', jl_scn='sim_nev_dr', st_scn='sim_nev_dr')
compare_test('test_attgt_matches_r[sim_nev_dr]', df)

PASS test_attgt_matches_r[sim_nev_dr] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.920911  0.920911  0.920911  0.920911 0.063919 0.063919 0.063919 0.063919       0.0      0.0 PASS
 2  3  1.987483  1.987483  1.987483  1.987483 0.064527 0.064527 0.064527 0.064527       0.0      0.0 PASS
 2  4  2.955212  2.955212  2.955212  2.955212 0.063079 0.063079 0.063079 0.063079       0.0      0.0 PASS
 3  2 -0.043291 -0.043291 -0.043291 -0.043291 0.065565 0.065565 0.065565 0.065565       0.0      0.0 PASS
 3  3  1.108047  1.108047  1.108047  1.108047 0.065224 0.065224 0.065224 0.065224       0.0      0.0 PASS
 3  4  2.059014  2.059014  2.059014  2.059014 0.065203 0.065203 0.065203 0.065203       0.0      0.0 PASS
 4  2  0.002329  0.002329  0.002329  0.002329 0.067775 0.067775 0.067775 0.067775       0.0      0.0 PASS
 4  3  0.061537  0.061537  0.061537  0.061537 0.066011 0.066011 0.066011 0.066011       0.0      0.0 PAS

## B. test_r_parity: Aggregated overall ATT and SE (5 tests)

### Test 6 — test_aggte_overall_matches_r[mpdta_nev_dr]

`mpdta`, DR, nevertreated, no covariates -- simple/dynamic/group/calendar overall

In [7]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_overall(res, r_scn='mpdta_nev_dr', jl_scn='mpdta_nev_dr', st_scn='dr_nev_nocov')
compare_test('test_aggte_overall_matches_r[mpdta_nev_dr]', df)

PASS test_aggte_overall_matches_r[mpdta_nev_dr] (4 cells)
       g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
  simple - -0.039951 -0.039951 -0.039951 -0.039951 0.012034 0.012034 0.012034 0.012034       0.0      0.0 PASS
 dynamic - -0.077240 -0.077240 -0.077240 -0.077240 0.019965 0.019965 0.019965 0.019965       0.0      0.0 PASS
   group - -0.031018 -0.031018 -0.031018 -0.031018 0.012446 0.012446 0.012446 0.012446       0.0      0.0 PASS
calendar - -0.041700 -0.041700 -0.041700 -0.041700 0.015972 0.015972 0.015972 0.015972       0.0      0.0 PASS


### Test 7 — test_aggte_overall_matches_r[mpdta_nyt_dr]

`mpdta`, DR, notyettreated, no covariates -- simple/dynamic/group/calendar overall

In [8]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_overall(res, r_scn='mpdta_nyt_dr', jl_scn='mpdta_nyt_dr', st_scn='dr_nyt_nocov')
compare_test('test_aggte_overall_matches_r[mpdta_nyt_dr]', df)

PASS test_aggte_overall_matches_r[mpdta_nyt_dr] (4 cells)
       g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
  simple - -0.039764 -0.039764 -0.039764 -0.039764 0.012052 0.012052 0.012052 0.012052       0.0      0.0 PASS
 dynamic - -0.077399 -0.077399 -0.077399 -0.077399 0.019560 0.019560 0.019560 0.019560       0.0      0.0 PASS
   group - -0.030462 -0.030462 -0.030462 -0.030462 0.012575 0.012575 0.012575 0.012575       0.0      0.0 PASS
calendar - -0.044267 -0.044267 -0.044267 -0.044267 0.015571 0.015571 0.015571 0.015571       0.0      0.0 PASS


### Test 8 — test_aggte_overall_matches_r[mpdta_nev_reg_cov]

`mpdta`, REG, nevertreated, covariates: lpop -- simple/dynamic/group/calendar overall

In [9]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~lpop', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_aggte_overall(res, r_scn='mpdta_nev_reg_cov', jl_scn='mpdta_nev_reg_cov', st_scn='reg_nev_cov')
compare_test('test_aggte_overall_matches_r[mpdta_nev_reg_cov]', df)

PASS test_aggte_overall_matches_r[mpdta_nev_reg_cov] (4 cells)
       g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
  simple - -0.041969 -0.041969 -0.041969 -0.041969 0.011445 0.011445 0.011445 0.011445       0.0      0.0 PASS
 dynamic - -0.080782 -0.080782 -0.080782 -0.080782 0.018746 0.018746 0.018746 0.018746       0.0      0.0 PASS
   group - -0.032929 -0.032929 -0.032929 -0.032929 0.011860 0.011860 0.011860 0.011860       0.0      0.0 PASS
calendar - -0.044532 -0.044532 -0.044532 -0.044532 0.014885 0.014885 0.014885 0.014885       0.0      0.0 PASS


### Test 9 — test_aggte_overall_matches_r[mpdta_nev_ipw]

`mpdta`, IPW, nevertreated, no covariates -- simple/dynamic/group/calendar overall

In [10]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='ipw', bstrap=False)
df = prep_aggte_overall(res, r_scn='mpdta_nev_ipw', jl_scn='mpdta_nev_ipw', st_scn='ipw_nev_nocov')
compare_test('test_aggte_overall_matches_r[mpdta_nev_ipw]', df)

PASS test_aggte_overall_matches_r[mpdta_nev_ipw] (4 cells)
       g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
  simple - -0.039951 -0.039951 -0.039951 -0.039951 0.012034 0.012034 0.012034 0.012034       0.0      0.0 PASS
 dynamic - -0.077240 -0.077240 -0.077240 -0.077240 0.019965 0.019965 0.019965 0.019965       0.0      0.0 PASS
   group - -0.031018 -0.031018 -0.031018 -0.031018 0.012446 0.012446 0.012446 0.012446       0.0      0.0 PASS
calendar - -0.041700 -0.041700 -0.041700 -0.041700 0.015972 0.015972 0.015972 0.015972       0.0      0.0 PASS


### Test 10 — test_aggte_overall_matches_r[sim_nev_dr]

`sim_data`, DR, nevertreated, covariates: X -- simple/dynamic/group/calendar overall

In [11]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_overall(res, r_scn='sim_nev_dr', jl_scn='sim_nev_dr', st_scn='sim_nev_dr')
compare_test('test_aggte_overall_matches_r[sim_nev_dr]', df)

PASS test_aggte_overall_matches_r[sim_nev_dr] (4 cells)
       g t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
  simple - 1.658345 1.658345 1.658345 1.658345 0.034294 0.034294 0.034294 0.034294       0.0      0.0 PASS
 dynamic - 1.990409 1.990409 1.990409 1.990409 0.038630 0.038630 0.038630 0.038630       0.0      0.0 PASS
   group - 1.488044 1.488044 1.488044 1.488044 0.034901 0.034901 0.034901 0.034901       0.0      0.0 PASS
calendar - 1.480802 1.480802 1.480802 1.480802 0.035727 0.035727 0.035727 0.035727       0.0      0.0 PASS


## C. test_r_parity: Per-event/group/calendar ATT(egt) and SE (15 tests)

### Test 11 — test_aggte_egt_matches_r[group-mpdta_nev_dr]

`mpdta`, DR, nevertreated, no covariates -- group aggregation per-egt + overall

In [12]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_dr', jl_scn='mpdta_nev_dr', agg_type='group', st_scn='dr_nev_nocov')
compare_test('test_aggte_egt_matches_r[group-mpdta_nev_dr]', df)

PASS test_aggte_egt_matches_r[group-mpdta_nev_dr] (4 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.031018 -0.031018 -0.031018 -0.031018 0.012446 0.012446 0.012446 0.012446       0.0      0.0 PASS
 g=2004 - -0.079749 -0.079749 -0.079749 -0.079749 0.026368 0.026368 0.026368 0.026368       0.0      0.0 PASS
 g=2006 - -0.022909 -0.022909 -0.022909 -0.022909 0.016703 0.016703 0.016703 0.016703       0.0      0.0 PASS
 g=2007 - -0.026054 -0.026054 -0.026054 -0.026054 0.016655 0.016655 0.016655 0.016655       0.0      0.0 PASS


### Test 12 — test_aggte_egt_matches_r[group-mpdta_nyt_dr]

`mpdta`, DR, notyettreated, no covariates -- group aggregation per-egt + overall

In [13]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nyt_dr', jl_scn='mpdta_nyt_dr', agg_type='group', st_scn='dr_nyt_nocov')
compare_test('test_aggte_egt_matches_r[group-mpdta_nyt_dr]', df)

PASS test_aggte_egt_matches_r[group-mpdta_nyt_dr] (4 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.030462 -0.030462 -0.030462 -0.030462 0.012575 0.012575 0.012575 0.012575       0.0      0.0 PASS
 g=2004 - -0.083694 -0.083694 -0.083694 -0.083694 0.025702 0.025702 0.025702 0.025702       0.0      0.0 PASS
 g=2006 - -0.018282 -0.018282 -0.018282 -0.018282 0.015922 0.015922 0.015922 0.015922       0.0      0.0 PASS
 g=2007 - -0.026054 -0.026054 -0.026054 -0.026054 0.016655 0.016655 0.016655 0.016655       0.0      0.0 PASS


### Test 13 — test_aggte_egt_matches_r[group-mpdta_nev_reg_cov]

`mpdta`, REG, nevertreated, covariates: lpop -- group aggregation per-egt + overall

In [14]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~lpop', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_reg_cov', jl_scn='mpdta_nev_reg_cov', agg_type='group', st_scn='reg_nev_cov')
compare_test('test_aggte_egt_matches_r[group-mpdta_nev_reg_cov]', df)

PASS test_aggte_egt_matches_r[group-mpdta_nev_reg_cov] (4 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.032929 -0.032929 -0.032929 -0.032929 0.011860 0.011860 0.011860 0.011860       0.0      0.0 PASS
 g=2004 - -0.085133 -0.085133 -0.085133 -0.085133 0.024251 0.024251 0.024251 0.024251       0.0      0.0 PASS
 g=2006 - -0.020385 -0.020385 -0.020385 -0.020385 0.017403 0.017403 0.017403 0.017403       0.0      0.0 PASS
 g=2007 - -0.028789 -0.028789 -0.028789 -0.028789 0.016168 0.016168 0.016168 0.016168       0.0      0.0 PASS


### Test 14 — test_aggte_egt_matches_r[group-mpdta_nev_ipw]

`mpdta`, IPW, nevertreated, no covariates -- group aggregation per-egt + overall

In [15]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='ipw', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_ipw', jl_scn='mpdta_nev_ipw', agg_type='group', st_scn='ipw_nev_nocov')
compare_test('test_aggte_egt_matches_r[group-mpdta_nev_ipw]', df)

PASS test_aggte_egt_matches_r[group-mpdta_nev_ipw] (4 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.031018 -0.031018 -0.031018 -0.031018 0.012446 0.012446 0.012446 0.012446       0.0      0.0 PASS
 g=2004 - -0.079749 -0.079749 -0.079749 -0.079749 0.026368 0.026368 0.026368 0.026368       0.0      0.0 PASS
 g=2006 - -0.022909 -0.022909 -0.022909 -0.022909 0.016703 0.016703 0.016703 0.016703       0.0      0.0 PASS
 g=2007 - -0.026054 -0.026054 -0.026054 -0.026054 0.016655 0.016655 0.016655 0.016655       0.0      0.0 PASS


### Test 15 — test_aggte_egt_matches_r[group-sim_nev_dr]

`sim_data`, DR, nevertreated, covariates: X -- group aggregation per-egt + overall

In [16]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='sim_nev_dr', jl_scn='sim_nev_dr', agg_type='group', st_scn='sim_nev_dr')
compare_test('test_aggte_egt_matches_r[group-sim_nev_dr]', df)

PASS test_aggte_egt_matches_r[group-sim_nev_dr] (4 cells)
      g t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - 1.488044 1.488044 1.488044 1.488044 0.034901 0.034901 0.034901 0.034901       0.0      0.0 PASS
    e=2 - 1.954535 1.954535 1.954535 1.954535 0.052134 0.052134 0.052134 0.052134       0.0      0.0 PASS
    e=3 - 1.583530 1.583530 1.583530 1.583530 0.056060 0.056060 0.056060 0.056060       0.0      0.0 PASS
    e=4 - 0.952316 0.952316 0.952316 0.952316 0.067021 0.067021 0.067021 0.067021       0.0      0.0 PASS


### Test 16 — test_aggte_egt_matches_r[dynamic-mpdta_nev_dr]

`mpdta`, DR, nevertreated, no covariates -- dynamic aggregation per-egt + overall

In [17]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_dr', jl_scn='mpdta_nev_dr', agg_type='dynamic', st_scn='dr_nev_nocov')
compare_test('test_aggte_egt_matches_r[dynamic-mpdta_nev_dr]', df)

PASS test_aggte_egt_matches_r[dynamic-mpdta_nev_dr] (8 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.077240 -0.077240 -0.077240 -0.077240 0.019965 0.019965 0.019965 0.019965       0.0      0.0 PASS
   e=-3 -  0.030507  0.030507  0.030507  0.030507 0.015034 0.015034 0.015034 0.015034       0.0      0.0 PASS
   e=-2 - -0.000563 -0.000563 -0.000563 -0.000563 0.013292 0.013292 0.013292 0.013292       0.0      0.0 PASS
   e=-1 - -0.024459 -0.024459 -0.024459 -0.024459 0.014236 0.014236 0.014236 0.014236       0.0      0.0 PASS
    e=0 - -0.019932 -0.019932 -0.019932 -0.019932 0.011826 0.011826 0.011826 0.011826       0.0      0.0 PASS
    e=1 - -0.050957 -0.050957 -0.050957 -0.050957 0.016893 0.016893 0.016893 0.016893       0.0      0.0 PASS
    e=2 - -0.137259 -0.137259 -0.137259 -0.137259 0.036436 0.036436 0.036436 0.036436       0.0      0.0 PASS
    e=3 - -0.100811 -0.100811 -0.100811 -0.100811 0.034359

### Test 17 — test_aggte_egt_matches_r[dynamic-mpdta_nyt_dr]

`mpdta`, DR, notyettreated, no covariates -- dynamic aggregation per-egt + overall

In [18]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nyt_dr', jl_scn='mpdta_nyt_dr', agg_type='dynamic', st_scn='dr_nyt_nocov')
compare_test('test_aggte_egt_matches_r[dynamic-mpdta_nyt_dr]', df)

PASS test_aggte_egt_matches_r[dynamic-mpdta_nyt_dr] (8 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.077399 -0.077399 -0.077399 -0.077399 0.019560 0.019560 0.019560 0.019560       0.0      0.0 PASS
   e=-3 -  0.029759  0.029759  0.029759  0.029759 0.014533 0.014533 0.014533 0.014533       0.0      0.0 PASS
   e=-2 - -0.002446 -0.002446 -0.002446 -0.002446 0.013120 0.013120 0.013120 0.013120       0.0      0.0 PASS
   e=-1 - -0.024269 -0.024269 -0.024269 -0.024269 0.014464 0.014464 0.014464 0.014464       0.0      0.0 PASS
    e=0 - -0.018922 -0.018922 -0.018922 -0.018922 0.012045 0.012045 0.012045 0.012045       0.0      0.0 PASS
    e=1 - -0.053589 -0.053589 -0.053589 -0.053589 0.016946 0.016946 0.016946 0.016946       0.0      0.0 PASS
    e=2 - -0.136274 -0.136274 -0.136274 -0.136274 0.035403 0.035403 0.035403 0.035403       0.0      0.0 PASS
    e=3 - -0.100811 -0.100811 -0.100811 -0.100811 0.034359

### Test 18 — test_aggte_egt_matches_r[dynamic-mpdta_nev_reg_cov]

`mpdta`, REG, nevertreated, covariates: lpop -- dynamic aggregation per-egt + overall

In [19]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~lpop', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_reg_cov', jl_scn='mpdta_nev_reg_cov', agg_type='dynamic', st_scn='reg_nev_cov')
compare_test('test_aggte_egt_matches_r[dynamic-mpdta_nev_reg_cov]', df)

PASS test_aggte_egt_matches_r[dynamic-mpdta_nev_reg_cov] (8 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.080782 -0.080782 -0.080782 -0.080782 0.018746 0.018746 0.018746 0.018746       0.0      0.0 PASS
   e=-3 -  0.026366  0.026366  0.026366  0.026366 0.014019 0.014019 0.014019 0.014019       0.0      0.0 PASS
   e=-2 - -0.004130 -0.004130 -0.004130 -0.004130 0.012918 0.012918 0.012918 0.012918       0.0      0.0 PASS
   e=-1 - -0.023465 -0.023465 -0.023465 -0.023465 0.014442 0.014442 0.014442 0.014442       0.0      0.0 PASS
    e=0 - -0.021147 -0.021147 -0.021147 -0.021147 0.011481 0.011481 0.011481 0.011481       0.0      0.0 PASS
    e=1 - -0.053356 -0.053356 -0.053356 -0.053356 0.016293 0.016293 0.016293 0.016293       0.0      0.0 PASS
    e=2 - -0.141080 -0.141080 -0.141080 -0.141080 0.034836 0.034836 0.034836 0.034836       0.0      0.0 PASS
    e=3 - -0.107544 -0.107544 -0.107544 -0.107544 0.0

### Test 19 — test_aggte_egt_matches_r[dynamic-mpdta_nev_ipw]

`mpdta`, IPW, nevertreated, no covariates -- dynamic aggregation per-egt + overall

In [20]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='ipw', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_ipw', jl_scn='mpdta_nev_ipw', agg_type='dynamic', st_scn='ipw_nev_nocov')
compare_test('test_aggte_egt_matches_r[dynamic-mpdta_nev_ipw]', df)

PASS test_aggte_egt_matches_r[dynamic-mpdta_nev_ipw] (8 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.077240 -0.077240 -0.077240 -0.077240 0.019965 0.019965 0.019965 0.019965       0.0      0.0 PASS
   e=-3 -  0.030507  0.030507  0.030507  0.030507 0.015034 0.015034 0.015034 0.015034       0.0      0.0 PASS
   e=-2 - -0.000563 -0.000563 -0.000563 -0.000563 0.013292 0.013292 0.013292 0.013292       0.0      0.0 PASS
   e=-1 - -0.024459 -0.024459 -0.024459 -0.024459 0.014236 0.014236 0.014236 0.014236       0.0      0.0 PASS
    e=0 - -0.019932 -0.019932 -0.019932 -0.019932 0.011826 0.011826 0.011826 0.011826       0.0      0.0 PASS
    e=1 - -0.050957 -0.050957 -0.050957 -0.050957 0.016893 0.016893 0.016893 0.016893       0.0      0.0 PASS
    e=2 - -0.137259 -0.137259 -0.137259 -0.137259 0.036436 0.036436 0.036436 0.036436       0.0      0.0 PASS
    e=3 - -0.100811 -0.100811 -0.100811 -0.100811 0.03435

### Test 20 — test_aggte_egt_matches_r[dynamic-sim_nev_dr]

`sim_data`, DR, nevertreated, covariates: X -- dynamic aggregation per-egt + overall

In [21]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='sim_nev_dr', jl_scn='sim_nev_dr', agg_type='dynamic', st_scn='sim_nev_dr')
compare_test('test_aggte_egt_matches_r[dynamic-sim_nev_dr]', df)

PASS test_aggte_egt_matches_r[dynamic-sim_nev_dr] (6 cells)
      g t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - 1.990409 1.990409 1.990409 1.990409 0.038630 0.038630 0.038630 0.038630       0.0      0.0 PASS
   e=-2 - 0.002329 0.002329 0.002329 0.002329 0.067775 0.067775 0.067775 0.067775       0.0      0.0 PASS
   e=-1 - 0.010510 0.010510 0.010510 0.010510 0.040277 0.040277 0.040277 0.040277       0.0      0.0 PASS
    e=0 - 0.992876 0.992876 0.992876 0.992876 0.030647 0.030647 0.030647 0.030647       0.0      0.0 PASS
    e=1 - 2.023139 2.023139 2.023139 2.023139 0.045557 0.045557 0.045557 0.045557       0.0      0.0 PASS
    e=2 - 2.955212 2.955212 2.955212 2.955212 0.063079 0.063079 0.063079 0.063079       0.0      0.0 PASS


### Test 21 — test_aggte_egt_matches_r[calendar-mpdta_nev_dr]

`mpdta`, DR, nevertreated, no covariates -- calendar aggregation per-egt + overall

In [22]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_dr', jl_scn='mpdta_nev_dr', agg_type='calendar', st_scn='dr_nev_nocov')
compare_test('test_aggte_egt_matches_r[calendar-mpdta_nev_dr]', df)

PASS test_aggte_egt_matches_r[calendar-mpdta_nev_dr] (5 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.041700 -0.041700 -0.041700 -0.041700 0.015972 0.015972 0.015972 0.015972       0.0      0.0 PASS
 g=2004 - -0.010503 -0.010503 -0.010503 -0.010503 0.023251 0.023251 0.023251 0.023251       0.0      0.0 PASS
 g=2005 - -0.070423 -0.070423 -0.070423 -0.070423 0.030985 0.030985 0.030985 0.030985       0.0      0.0 PASS
 g=2006 - -0.048816 -0.048816 -0.048816 -0.048816 0.020126 0.020126 0.020126 0.020126       0.0      0.0 PASS
 g=2007 - -0.037059 -0.037059 -0.037059 -0.037059 0.013747 0.013747 0.013747 0.013747       0.0      0.0 PASS


### Test 22 — test_aggte_egt_matches_r[calendar-mpdta_nyt_dr]

`mpdta`, DR, notyettreated, no covariates -- calendar aggregation per-egt + overall

In [23]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nyt_dr', jl_scn='mpdta_nyt_dr', agg_type='calendar', st_scn='dr_nyt_nocov')
compare_test('test_aggte_egt_matches_r[calendar-mpdta_nyt_dr]', df)

PASS test_aggte_egt_matches_r[calendar-mpdta_nyt_dr] (5 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
overall - -0.044267 -0.044267 -0.044267 -0.044267 0.015571 0.015571 0.015571 0.015571 0.000000e+00      0.0 PASS
 g=2004 - -0.019372 -0.019372 -0.019372 -0.019372 0.022310 0.022310 0.022310 0.022310 1.000000e-07      0.0 PASS
 g=2005 - -0.078319 -0.078319 -0.078319 -0.078319 0.030390 0.030390 0.030390 0.030390 1.000000e-07      0.0 PASS
 g=2006 - -0.042318 -0.042318 -0.042318 -0.042318 0.019056 0.019056 0.019056 0.019056 0.000000e+00      0.0 PASS
 g=2007 - -0.037059 -0.037059 -0.037059 -0.037059 0.013747 0.013747 0.013747 0.013747 0.000000e+00      0.0 PASS


### Test 23 — test_aggte_egt_matches_r[calendar-mpdta_nev_reg_cov]

`mpdta`, REG, nevertreated, covariates: lpop -- calendar aggregation per-egt + overall

In [24]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~lpop', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_reg_cov', jl_scn='mpdta_nev_reg_cov', agg_type='calendar', st_scn='reg_nev_cov')
compare_test('test_aggte_egt_matches_r[calendar-mpdta_nev_reg_cov]', df)

PASS test_aggte_egt_matches_r[calendar-mpdta_nev_reg_cov] (5 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.044532 -0.044532 -0.044532 -0.044532 0.014885 0.014885 0.014885 0.014885       0.0      0.0 PASS
 g=2004 - -0.014911 -0.014911 -0.014911 -0.014911 0.022056 0.022056 0.022056 0.022056       0.0      0.0 PASS
 g=2005 - -0.076996 -0.076996 -0.076996 -0.076996 0.028360 0.028360 0.028360 0.028360       0.0      0.0 PASS
 g=2006 - -0.046516 -0.046516 -0.046516 -0.046516 0.020998 0.020998 0.020998 0.020998       0.0      0.0 PASS
 g=2007 - -0.039705 -0.039705 -0.039705 -0.039705 0.012866 0.012866 0.012866 0.012866       0.0      0.0 PASS


### Test 24 — test_aggte_egt_matches_r[calendar-mpdta_nev_ipw]

`mpdta`, IPW, nevertreated, no covariates -- calendar aggregation per-egt + overall

In [25]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated'
            ).fit(est_method='ipw', bstrap=False)
df = prep_aggte_egt(res, r_scn='mpdta_nev_ipw', jl_scn='mpdta_nev_ipw', agg_type='calendar', st_scn='ipw_nev_nocov')
compare_test('test_aggte_egt_matches_r[calendar-mpdta_nev_ipw]', df)

PASS test_aggte_egt_matches_r[calendar-mpdta_nev_ipw] (5 cells)
      g t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - -0.041700 -0.041700 -0.041700 -0.041700 0.015972 0.015972 0.015972 0.015972       0.0      0.0 PASS
 g=2004 - -0.010503 -0.010503 -0.010503 -0.010503 0.023251 0.023251 0.023251 0.023251       0.0      0.0 PASS
 g=2005 - -0.070423 -0.070423 -0.070423 -0.070423 0.030985 0.030985 0.030985 0.030985       0.0      0.0 PASS
 g=2006 - -0.048816 -0.048816 -0.048816 -0.048816 0.020126 0.020126 0.020126 0.020126       0.0      0.0 PASS
 g=2007 - -0.037059 -0.037059 -0.037059 -0.037059 0.013747 0.013747 0.013747 0.013747       0.0      0.0 PASS


### Test 25 — test_aggte_egt_matches_r[calendar-sim_nev_dr]

`sim_data`, DR, nevertreated, covariates: X -- calendar aggregation per-egt + overall

In [26]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_aggte_egt(res, r_scn='sim_nev_dr', jl_scn='sim_nev_dr', agg_type='calendar', st_scn='sim_nev_dr')
compare_test('test_aggte_egt_matches_r[calendar-sim_nev_dr]', df)

PASS test_aggte_egt_matches_r[calendar-sim_nev_dr] (4 cells)
      g t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
overall - 1.480802 1.480802 1.480802 1.480802 0.035727 0.035727 0.035727 0.035727       0.0      0.0 PASS
    e=2 - 0.920911 0.920911 0.920911 0.920911 0.063919 0.063919 0.063919 0.063919       0.0      0.0 PASS
    e=3 - 1.549114 1.549114 1.549114 1.549114 0.052176 0.052176 0.052176 0.052176       0.0      0.0 PASS
    e=4 - 1.972380 1.972380 1.972380 1.972380 0.048774 0.048774 0.048774 0.048774       0.0      0.0 PASS


## D. test_r_parity: fix_weights (time-varying weights) (4 tests)

### Test 26 — test_fix_weights_matches_r[none]

`mpdta_tvw`, REG, nevertreated, weights=wt, fix_weights=none

In [27]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_tvw, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated',
            weights_name='wt', fix_weights=None
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='fix_weights_none', st_scn='fix_weights_none',
                r_df=r_fixwt, r_filt={'fix_weights': 'none'})
compare_test('test_fix_weights_matches_r[none]', df)

PASS test_fix_weights_matches_r[none] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.011575 -0.011575 -0.011575 -0.011575 0.023913 0.023913 0.023913 0.023913 0.000000e+00      0.0 PASS
2004 2005 -0.067484 -0.067484 -0.067484 -0.067484 0.031717 0.031717 0.031717 0.031717 0.000000e+00      0.0 PASS
2004 2006 -0.136539 -0.136539 -0.136539 -0.136539 0.036779 0.036779 0.036779 0.036779 0.000000e+00      0.0 PASS
2004 2007 -0.097612 -0.097612 -0.097612 -0.097612 0.034799 0.034799 0.034799 0.034799 0.000000e+00      0.0 PASS
2006 2004  0.008036  0.008036  0.008036  0.008036 0.023299 0.023299 0.023299 0.023299 1.000000e-07      0.0 PASS
2006 2005 -0.002333 -0.002333 -0.002333 -0.002333 0.019264 0.019264 0.019264 0.019264 0.000000e+00      0.0 PASS
2006 2006 -0.005478 -0.005478 -0.005478 -0.005478 0.017621 0.017621 0.017621 0.017621 0.000000e+00      0.0 PASS
2006 2007 -0.040752 -0.040752 -0.040752 -0.0407

### Test 27 — test_fix_weights_matches_r[base_period]

`mpdta_tvw`, REG, nevertreated, weights=wt, fix_weights=base_period

In [28]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_tvw, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated',
            weights_name='wt', fix_weights='base_period'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='fix_weights_base', st_scn='fix_weights_base',
                r_df=r_fixwt, r_filt={'fix_weights': 'base_period'})
compare_test('test_fix_weights_matches_r[base_period]', df)

PASS test_fix_weights_matches_r[base_period] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.011575 -0.011575 -0.011575 -0.011575 0.023913 0.023913 0.023913 0.023913 0.000000e+00      0.0 PASS
2004 2005 -0.067484 -0.067484 -0.067484 -0.067484 0.031717 0.031717 0.031717 0.031717 0.000000e+00      0.0 PASS
2004 2006 -0.136539 -0.136539 -0.136539 -0.136539 0.036779 0.036779 0.036779 0.036779 0.000000e+00      0.0 PASS
2004 2007 -0.097612 -0.097612 -0.097612 -0.097612 0.034799 0.034799 0.034799 0.034799 0.000000e+00      0.0 PASS
2006 2004  0.007814  0.007814  0.007814  0.007814 0.023298 0.023298 0.023298 0.023298 1.000000e-07      0.0 PASS
2006 2005 -0.002364 -0.002364 -0.002364 -0.002364 0.019284 0.019284 0.019284 0.019284 0.000000e+00      0.0 PASS
2006 2006 -0.005478 -0.005478 -0.005478 -0.005478 0.017621 0.017621 0.017621 0.017621 0.000000e+00      0.0 PASS
2006 2007 -0.040752 -0.040752 -0.040752 

### Test 28 — test_fix_weights_matches_r[first_period]

`mpdta_tvw`, REG, nevertreated, weights=wt, fix_weights=first_period

In [29]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_tvw, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated',
            weights_name='wt', fix_weights='first_period'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='fix_weights_first', st_scn='fix_weights_first',
                r_df=r_fixwt, r_filt={'fix_weights': 'first_period'})
compare_test('test_fix_weights_matches_r[first_period]', df)

PASS test_fix_weights_matches_r[first_period] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.011575 -0.011575 -0.011575 -0.011575 0.023913 0.023913 0.023913 0.023913 0.000000e+00      0.0 PASS
2004 2005 -0.067484 -0.067484 -0.067484 -0.067484 0.031717 0.031717 0.031717 0.031717 0.000000e+00      0.0 PASS
2004 2006 -0.136539 -0.136539 -0.136539 -0.136539 0.036779 0.036779 0.036779 0.036779 0.000000e+00      0.0 PASS
2004 2007 -0.097612 -0.097612 -0.097612 -0.097612 0.034799 0.034799 0.034799 0.034799 0.000000e+00      0.0 PASS
2006 2004  0.008036  0.008036  0.008036  0.008036 0.023299 0.023299 0.023299 0.023299 1.000000e-07      0.0 PASS
2006 2005 -0.002296 -0.002296 -0.002296 -0.002296 0.019242 0.019242 0.019242 0.019242 0.000000e+00      0.0 PASS
2006 2006 -0.005632 -0.005632 -0.005632 -0.005632 0.017605 0.017605 0.017605 0.017605 0.000000e+00      0.0 PASS
2006 2007 -0.040673 -0.040673 -0.040673

### Test 29 — test_fix_weights_matches_r[varying]

`mpdta_tvw`, REG, nevertreated, weights=wt, fix_weights=varying

In [30]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_tvw, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', xformla='lemp~1', control_group='nevertreated',
            weights_name='wt', fix_weights='varying'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='fix_weights_varying', st_scn='fix_weights_varying',
                r_df=r_fixwt, r_filt={'fix_weights': 'varying'})
compare_test('test_fix_weights_matches_r[varying]', df)

PASS test_fix_weights_matches_r[varying] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.011433 -0.011433 -0.011433 -0.011433 0.023744 0.023744 0.023744 0.023744 0.000000e+00      0.0 PASS
2004 2005 -0.067812 -0.067812 -0.067812 -0.067812 0.032058 0.032058 0.032058 0.032058 0.000000e+00      0.0 PASS
2004 2006 -0.136538 -0.136538 -0.136538 -0.136538 0.037154 0.037154 0.037154 0.037154 0.000000e+00      0.0 PASS
2004 2007 -0.098247 -0.098247 -0.098247 -0.098247 0.036706 0.036706 0.036706 0.036706 0.000000e+00      0.0 PASS
2006 2004  0.007413  0.007413  0.007413  0.007413 0.023277 0.023277 0.023277 0.023277 1.000000e-07      0.0 PASS
2006 2005 -0.002896 -0.002896 -0.002896 -0.002896 0.019316 0.019316 0.019316 0.019316 1.000000e-07      0.0 PASS
2006 2006 -0.005902 -0.005902 -0.005902 -0.005902 0.017662 0.017662 0.017662 0.017662 0.000000e+00      0.0 PASS
2006 2007 -0.041720 -0.041720 -0.041720 -0.0

## E. test_r_parity: Factor (categorical) covariate (2 tests)

### Test 30 — test_factor_covariate_matches_r[False]

`factor_cov`, REG, nevertreated, xformla=Y~C(cat), faster_mode=False

In [31]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=factor_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~C(cat)', control_group='nevertreated',
            faster_mode=False
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='factor_cov', st_scn='factor_cov', r_df=r_factor)
compare_test('test_factor_covariate_matches_r[False]', df)

PASS test_factor_covariate_matches_r[False] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.926395  0.926395  0.926395  0.926395 0.095216 0.095216 0.095216 0.095216       0.0      0.0 PASS
 2  3  1.859478  1.859478  1.859478  1.859478 0.096255 0.096255 0.096255 0.096255       0.0      0.0 PASS
 2  4  2.991930  2.991930  2.991930  2.991930 0.090874 0.090874 0.090874 0.090874       0.0      0.0 PASS
 3  2 -0.157837 -0.157837 -0.157837 -0.157837 0.095232 0.095232 0.095232 0.095232       0.0      0.0 PASS
 3  3  0.910926  0.910926  0.910926  0.910926 0.088641 0.088641 0.088641 0.088641       0.0      0.0 PASS
 3  4  2.165231  2.165231  2.165231  2.165231 0.092373 0.092373 0.092373 0.092373       0.0      0.0 PASS
 4  2 -0.061912 -0.061912 -0.061912 -0.061912 0.094473 0.094473 0.094473 0.094473       0.0      0.0 PASS
 4  3  0.005624  0.005624  0.005624  0.005624 0.087848 0.087848 0.087848 0.087848       0.0      0

### Test 31 — test_factor_covariate_matches_r[True]

`factor_cov`, REG, nevertreated, xformla=Y~C(cat), faster_mode=True

In [32]:
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=factor_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~C(cat)', control_group='nevertreated',
            faster_mode=True
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='factor_cov', st_scn='factor_cov', r_df=r_factor)
compare_test('test_factor_covariate_matches_r[True]', df)

PASS test_factor_covariate_matches_r[True] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.926395  0.926395  0.926395  0.926395 0.095216 0.095216 0.095216 0.095216       0.0      0.0 PASS
 2  3  1.859478  1.859478  1.859478  1.859478 0.096255 0.096255 0.096255 0.096255       0.0      0.0 PASS
 2  4  2.991930  2.991930  2.991930  2.991930 0.090874 0.090874 0.090874 0.090874       0.0      0.0 PASS
 3  2 -0.157837 -0.157837 -0.157837 -0.157837 0.095232 0.095232 0.095232 0.095232       0.0      0.0 PASS
 3  3  0.910926  0.910926  0.910926  0.910926 0.088641 0.088641 0.088641 0.088641       0.0      0.0 PASS
 3  4  2.165231  2.165231  2.165231  2.165231 0.092373 0.092373 0.092373 0.092373       0.0      0.0 PASS
 4  2 -0.061912 -0.061912 -0.061912 -0.061912 0.094473 0.094473 0.094473 0.094473       0.0      0.0 PASS
 4  3  0.005624  0.005624  0.005624  0.005624 0.087848 0.087848 0.087848 0.087848       0.0      0.

## F. test_r_parity: Gap scenarios (RC, universal, anticipation, weighted, clustered) (5 tests)

### Test 32 — test_gap_scenarios_match_r[rc]

`mpdta_extra`, REG, nevertreated, panel=False

In [33]:
ctor_kw = dict(panel=False)
fit_kw  = dict(base_period='varying')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_extra, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', control_group='nevertreated', **ctor_kw
            ).fit(est_method='reg', bstrap=False, **fit_kw)
df = prep_attgt(res, jl_scn='rc', st_scn='rc',
                r_df=r_gaps, r_filt={'scenario': 'rc'})
compare_test('test_gap_scenarios_match_r[rc]', df)

PASS test_gap_scenarios_match_r[rc] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.010503 -0.010503 -0.010503 -0.010503 0.475830 0.475830 0.475830 0.475829 0.000000e+00      0.0 PASS
2004 2005 -0.070423 -0.070423 -0.070423 -0.070423 0.482270 0.482270 0.482270 0.482270 0.000000e+00      0.0 PASS
2004 2006 -0.137259 -0.137259 -0.137259 -0.137259 0.485621 0.485621 0.485621 0.485621 0.000000e+00      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.478955 0.478955 0.478955 0.478955 0.000000e+00      0.0 PASS
2006 2004  0.006520  0.006520  0.006520  0.006520 0.304105 0.304105 0.304105 0.304105 1.000000e-07      0.0 PASS
2006 2005 -0.002751 -0.002751 -0.002751 -0.002751 0.306865 0.306865 0.306865 0.306865 0.000000e+00      0.0 PASS
2006 2006 -0.004595 -0.004595 -0.004595 -0.004595 0.311269 0.311269 0.311269 0.311269 0.000000e+00      0.0 PASS
2006 2007 -0.041224 -0.041224 -0.041224 -0.041224

### Test 33 — test_gap_scenarios_match_r[universal]

`mpdta_extra`, REG, nevertreated, base_period='universal'

In [34]:
ctor_kw = dict()
fit_kw  = dict(base_period='universal')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_extra, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', control_group='nevertreated', **ctor_kw
            ).fit(est_method='reg', bstrap=False, **fit_kw)
df = prep_attgt(res, jl_scn='universal', st_scn='universal',
                r_df=r_gaps, r_filt={'scenario': 'universal'})
compare_test('test_gap_scenarios_match_r[universal]', df)

PASS test_gap_scenarios_match_r[universal] (15 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
2004 2003  0.000000  0.000000  0.000000  0.000000      NaN      NaN      NaN      NaN       0.0      NaN PASS
2004 2004 -0.010503 -0.010503 -0.010503 -0.010503 0.023251 0.023251 0.023251 0.023251       0.0      0.0 PASS
2004 2005 -0.070423 -0.070423 -0.070423 -0.070423 0.030985 0.030985 0.030985 0.030985       0.0      0.0 PASS
2004 2006 -0.137259 -0.137259 -0.137259 -0.137259 0.036436 0.036436 0.036436 0.036436       0.0      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.034359 0.034359 0.034359 0.034359       0.0      0.0 PASS
2006 2003 -0.003769 -0.003769 -0.003769 -0.003769 0.031342 0.031342 0.031342 0.031342       0.0      0.0 PASS
2006 2004  0.002751  0.002751  0.002751  0.002751 0.019559 0.019559 0.019559 0.019559       0.0      0.0 PASS
2006 2005  0.000000  0.000000  0.000000  0.000000      NaN      Na

### Test 34 — test_gap_scenarios_match_r[anticipation1]

`mpdta_extra`, REG, nevertreated, anticipation=1

In [35]:
ctor_kw = dict(anticipation=1)
fit_kw  = dict(base_period='varying')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_extra, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', control_group='nevertreated', **ctor_kw
            ).fit(est_method='reg', bstrap=False, **fit_kw)
df = prep_attgt(res, jl_scn='anticipation1', st_scn='anticipation1',
                r_df=r_gaps, r_filt={'scenario': 'anticipation1'})
compare_test('test_gap_scenarios_match_r[anticipation1]', df)

PASS test_gap_scenarios_match_r[anticipation1] (8 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2006 2004  0.006520  0.006520  0.006520  0.006520 0.023327 0.023327 0.023327 0.023327 1.000000e-07      0.0 PASS
2006 2005 -0.002751 -0.002751 -0.002751 -0.002751 0.019559 0.019559 0.019559 0.019559 0.000000e+00      0.0 PASS
2006 2006 -0.007345 -0.007345 -0.007345 -0.007345 0.022943 0.022943 0.022943 0.022943 0.000000e+00      0.0 PASS
2006 2007 -0.043975 -0.043975 -0.043975 -0.043975 0.026579 0.026579 0.026579 0.026579 0.000000e+00      0.0 PASS
2007 2004  0.030507  0.030507  0.030507  0.030507 0.015034 0.015034 0.015034 0.015034 0.000000e+00      0.0 PASS
2007 2005 -0.002726 -0.002726 -0.002726 -0.002726 0.016396 0.016396 0.016396 0.016396 0.000000e+00      0.0 PASS
2007 2006 -0.031087 -0.031087 -0.031087 -0.031087 0.017878 0.017878 0.017878 0.017878 0.000000e+00      0.0 PASS
2007 2007 -0.057141 -0.057141 -0.057141

### Test 35 — test_gap_scenarios_match_r[weighted]

`mpdta_extra`, REG, nevertreated, weights_name='wt'

In [36]:
ctor_kw = dict(weights_name='wt')
fit_kw  = dict(base_period='varying')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_extra, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', control_group='nevertreated', **ctor_kw
            ).fit(est_method='reg', bstrap=False, **fit_kw)
df = prep_attgt(res, jl_scn='weighted', st_scn='weighted',
                r_df=r_gaps, r_filt={'scenario': 'weighted'})
compare_test('test_gap_scenarios_match_r[weighted]', df)

PASS test_gap_scenarios_match_r[weighted] (12 cells)


   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.011202 -0.011202 -0.011202 -0.011202 0.023705 0.023705 0.023705 0.023705 0.000000e+00      0.0 PASS
2004 2005 -0.067266 -0.067266 -0.067266 -0.067266 0.031282 0.031282 0.031282 0.031282 1.000000e-07      0.0 PASS
2004 2006 -0.133247 -0.133247 -0.133247 -0.133247 0.035895 0.035895 0.035895 0.035895 0.000000e+00      0.0 PASS
2004 2007 -0.096507 -0.096507 -0.096507 -0.096507 0.036590 0.036590 0.036590 0.036590 0.000000e+00      0.0 PASS
2006 2004  0.002853  0.002853  0.002853  0.002852 0.025116 0.025116 0.025116 0.025116 1.000000e-07      0.0 PASS
2006 2005 -0.004175 -0.004175 -0.004175 -0.004175 0.020781 0.020781 0.020781 0.020781 0.000000e+00      0.0 PASS
2006 2006 -0.004407 -0.004407 -0.004407 -0.004407 0.018122 0.018122 0.018122 0.018122 0.000000e+00      0.0 PASS
2006 2007 -0.041767 -0.041767 -0.041767 -0.041767 0.021875 0.021875 0.021875 0.021875 0.000000e

### Test 36 — test_gap_scenarios_match_r[clustered]

`mpdta_extra`, REG, nevertreated, clustervar='clust'

In [37]:
ctor_kw = dict(clustervar='clust')
fit_kw  = dict(base_period='varying')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=mpdta_extra, yname='lemp', tname='year', idname='countyreal',
            gname='first.treat', control_group='nevertreated', **ctor_kw
            ).fit(est_method='reg', bstrap=False, **fit_kw)
df = prep_attgt(res, jl_scn='clustered', st_scn='clustered',
                r_df=r_gaps, r_filt={'scenario': 'clustered'})
compare_test('test_gap_scenarios_match_r[clustered]', df)

PASS test_gap_scenarios_match_r[clustered] (12 cells)
   g    t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se     att_diff  se_diff   ok
2004 2004 -0.010503 -0.010503 -0.010503 -0.010503 0.012134 0.012134 0.012134 0.012134 0.000000e+00      0.0 PASS
2004 2005 -0.070423 -0.070423 -0.070423 -0.070423 0.014510 0.014510 0.014510 0.014510 0.000000e+00      0.0 PASS
2004 2006 -0.137259 -0.137259 -0.137259 -0.137259 0.023202 0.023202 0.023202 0.023202 0.000000e+00      0.0 PASS
2004 2007 -0.100811 -0.100811 -0.100811 -0.100811 0.020798 0.020798 0.020798 0.020798 0.000000e+00      0.0 PASS
2006 2004  0.006520  0.006520  0.006520  0.006520 0.035848 0.035848 0.035848 0.035848 1.000000e-07      0.0 PASS
2006 2005 -0.002751 -0.002751 -0.002751 -0.002751 0.020837 0.020837 0.020837 0.020837 0.000000e+00      0.0 PASS
2006 2006 -0.004595 -0.004595 -0.004595 -0.004595 0.020284 0.020284 0.020284 0.020284 0.000000e+00      0.0 PASS
2006 2007 -0.041224 -0.041224 -0.041224 -0

## G. test_sim_parity: Simulated data ATT(g,t) and SE (24 tests)

### Test 37 — test_sim_attgt_matches_r[tp2_const-nevertreated-dr]

`tp2_const`, DR, nevertreated, covariates: X

In [38]:
_sim_data = load_sim('tp2_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp2_const_nevertreated_dr', st_scn='sim_tp2_const_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp2_const', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp2_const-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp2_const-nevertreated-dr] (1 cells)
 g  t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 1.035871 1.035871 1.035871 1.035871 0.039668 0.039668 0.039668 0.039668       0.0      0.0 PASS


### Test 38 — test_sim_attgt_matches_r[tp2_const-nevertreated-reg]

`tp2_const`, REG, nevertreated, covariates: X

In [39]:
_sim_data = load_sim('tp2_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp2_const_nevertreated_reg', st_scn='sim_tp2_const_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp2_const', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp2_const-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp2_const-nevertreated-reg] (1 cells)
 g  t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 1.035889 1.035889 1.035889 1.035889 0.039667 0.039667 0.039667 0.039667       0.0      0.0 PASS


### Test 39 — test_sim_attgt_matches_r[tp2_const-notyettreated-dr]

`tp2_const`, DR, notyettreated, covariates: X

In [40]:
_sim_data = load_sim('tp2_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp2_const_notyettreated_dr', st_scn='sim_tp2_const_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp2_const', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp2_const-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp2_const-notyettreated-dr] (1 cells)
 g  t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 1.035871 1.035871 1.035871 1.035871 0.039668 0.039668 0.039668 0.039668       0.0      0.0 PASS


### Test 40 — test_sim_attgt_matches_r[tp2_const-notyettreated-reg]

`tp2_const`, REG, notyettreated, covariates: X

In [41]:
_sim_data = load_sim('tp2_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp2_const_notyettreated_reg', st_scn='sim_tp2_const_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp2_const', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp2_const-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp2_const-notyettreated-reg] (1 cells)
 g  t    R_att   Py_att   Jl_att   St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 1.035889 1.035889 1.035889 1.035889 0.039667 0.039667 0.039667 0.039667       0.0      0.0 PASS


### Test 41 — test_sim_attgt_matches_r[tp4_const-nevertreated-dr]

`tp4_const`, DR, nevertreated, covariates: X

In [42]:
_sim_data = load_sim('tp4_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_const_nevertreated_dr', st_scn='sim_tp4_const_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp4_const', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp4_const-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp4_const-nevertreated-dr] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.890957  0.890957  0.890957  0.890957 0.126757 0.126757 0.126757 0.126757       0.0      0.0 PASS
 2  3  1.057843  1.057843  1.057843  1.057843 0.128590 0.128590 0.128590 0.128590       0.0      0.0 PASS
 2  4  1.109397  1.109397  1.109397  1.109397 0.123131 0.123131 0.123131 0.123131       0.0      0.0 PASS
 3  2  0.017807  0.017807  0.017807  0.017807 0.125367 0.125367 0.125367 0.125367       0.0      0.0 PASS
 3  3  1.052928  1.052928  1.052928  1.052928 0.132547 0.132547 0.132547 0.132547       0.0      0.0 PASS
 3  4  0.988476  0.988476  0.988476  0.988476 0.127841 0.127841 0.127841 0.127841       0.0      0.0 PASS
 4  2 -0.067942 -0.067942 -0.067942 -0.067942 0.127966 0.127966 0.127966 0.127966       0.0      0.0 PASS
 4  3  0.093095  0.093095  0.093095  0.093095 0.137329 0.137329 0.137329 0.137329    

### Test 42 — test_sim_attgt_matches_r[tp4_const-nevertreated-reg]

`tp4_const`, REG, nevertreated, covariates: X

In [43]:
_sim_data = load_sim('tp4_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_const_nevertreated_reg', st_scn='sim_tp4_const_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp4_const', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp4_const-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp4_const-nevertreated-reg] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.891128  0.891128  0.891128  0.891128 0.126761 0.126761 0.126761 0.126761       0.0      0.0 PASS
 2  3  1.058682  1.058682  1.058682  1.058682 0.128585 0.128585 0.128585 0.128585       0.0      0.0 PASS
 2  4  1.109256  1.109256  1.109256  1.109256 0.123242 0.123242 0.123242 0.123242       0.0      0.0 PASS
 3  2  0.017855  0.017855  0.017855  0.017855 0.125354 0.125354 0.125354 0.125354       0.0      0.0 PASS
 3  3  1.053077  1.053077  1.053077  1.053077 0.132595 0.132595 0.132595 0.132595       0.0      0.0 PASS
 3  4  0.988394  0.988394  0.988394  0.988394 0.127826 0.127826 0.127826 0.127826       0.0      0.0 PASS
 4  2 -0.067778 -0.067778 -0.067778 -0.067778 0.127959 0.127959 0.127959 0.127959       0.0      0.0 PASS
 4  3  0.093725  0.093725  0.093725  0.093725 0.137485 0.137485 0.137485 0.137485   

### Test 43 — test_sim_attgt_matches_r[tp4_const-notyettreated-dr]

`tp4_const`, DR, notyettreated, covariates: X

In [44]:
_sim_data = load_sim('tp4_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_const_notyettreated_dr', st_scn='sim_tp4_const_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp4_const', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp4_const-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp4_const-notyettreated-dr] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.917027  0.917027  0.917027  0.917027 0.105204 0.105204 0.105204 0.105204       0.0      0.0 PASS
 2  3  1.046286  1.046286  1.046286  1.046286 0.113034 0.113034 0.113034 0.113034       0.0      0.0 PASS
 2  4  1.109397  1.109397  1.109397  1.109397 0.123131 0.123131 0.123131 0.123131       0.0      0.0 PASS
 3  2  0.051470  0.051470  0.051470  0.051470 0.109457 0.109457 0.109457 0.109457       0.0      0.0 PASS
 3  3  1.006461  1.006461  1.006461  1.006461 0.113110 0.113110 0.113110 0.113110       0.0      0.0 PASS
 3  4  0.988476  0.988476  0.988476  0.988476 0.127841 0.127841 0.127841 0.127841       0.0      0.0 PASS
 4  2 -0.066393 -0.066393 -0.066393 -0.066393 0.111641 0.111641 0.111641 0.111641       0.0      0.0 PASS
 4  3  0.093095  0.093095  0.093095  0.093095 0.137329 0.137329 0.137329 0.137329   

### Test 44 — test_sim_attgt_matches_r[tp4_const-notyettreated-reg]

`tp4_const`, REG, notyettreated, covariates: X

In [45]:
_sim_data = load_sim('tp4_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_const_notyettreated_reg', st_scn='sim_tp4_const_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp4_const', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp4_const-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp4_const-notyettreated-reg] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.917030  0.917030  0.917030  0.917030 0.105203 0.105203 0.105203 0.105203       0.0      0.0 PASS
 2  3  1.046322  1.046322  1.046322  1.046322 0.113034 0.113034 0.113034 0.113034       0.0      0.0 PASS
 2  4  1.109256  1.109256  1.109256  1.109256 0.123242 0.123242 0.123242 0.123242       0.0      0.0 PASS
 3  2  0.051470  0.051470  0.051470  0.051470 0.109456 0.109456 0.109456 0.109456       0.0      0.0 PASS
 3  3  1.006461  1.006461  1.006461  1.006461 0.113109 0.113109 0.113109 0.113109       0.0      0.0 PASS
 3  4  0.988394  0.988394  0.988394  0.988394 0.127826 0.127826 0.127826 0.127826       0.0      0.0 PASS
 4  2 -0.066136 -0.066136 -0.066136 -0.066136 0.111627 0.111627 0.111627 0.111627       0.0      0.0 PASS
 4  3  0.093725  0.093725  0.093725  0.093725 0.137485 0.137485 0.137485 0.137485  

### Test 45 — test_sim_attgt_matches_r[tp4_dyn-nevertreated-dr]

`tp4_dyn`, DR, nevertreated, covariates: X

In [46]:
_sim_data = load_sim('tp4_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_dyn_nevertreated_dr', st_scn='sim_tp4_dyn_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp4_dyn', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp4_dyn-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp4_dyn-nevertreated-dr] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.008317  1.008317  1.008317  1.008317 0.089854 0.089854 0.089854 0.089854       0.0      0.0 PASS
 2  3  2.013886  2.013886  2.013886  2.013886 0.087389 0.087389 0.087389 0.087389       0.0      0.0 PASS
 2  4  2.998124  2.998124  2.998124  2.998124 0.089488 0.089488 0.089488 0.089488       0.0      0.0 PASS
 3  2 -0.081417 -0.081417 -0.081417 -0.081417 0.089040 0.089040 0.089040 0.089040       0.0      0.0 PASS
 3  3  1.046460  1.046460  1.046460  1.046460 0.086088 0.086088 0.086088 0.086088       0.0      0.0 PASS
 3  4  2.121167  2.121167  2.121167  2.121167 0.089275 0.089275 0.089275 0.089275       0.0      0.0 PASS
 4  2 -0.030000 -0.030000 -0.030000 -0.030000 0.090195 0.090195 0.090195 0.090195       0.0      0.0 PASS
 4  3 -0.088217 -0.088217 -0.088217 -0.088217 0.089893 0.089893 0.089893 0.089893      

### Test 46 — test_sim_attgt_matches_r[tp4_dyn-nevertreated-reg]

`tp4_dyn`, REG, nevertreated, covariates: X

In [47]:
_sim_data = load_sim('tp4_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_dyn_nevertreated_reg', st_scn='sim_tp4_dyn_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp4_dyn', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp4_dyn-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp4_dyn-nevertreated-reg] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.008132  1.008132  1.008132  1.008132 0.089869 0.089869 0.089869 0.089869       0.0      0.0 PASS
 2  3  2.013492  2.013492  2.013492  2.013492 0.087391 0.087391 0.087391 0.087391       0.0      0.0 PASS
 2  4  2.997542  2.997542  2.997542  2.997542 0.089460 0.089460 0.089460 0.089460       0.0      0.0 PASS
 3  2 -0.081552 -0.081552 -0.081552 -0.081552 0.089057 0.089057 0.089057 0.089057       0.0      0.0 PASS
 3  3  1.046304  1.046304  1.046304  1.046304 0.086095 0.086095 0.086095 0.086095       0.0      0.0 PASS
 3  4  2.120878  2.120878  2.120878  2.120878 0.089273 0.089273 0.089273 0.089273       0.0      0.0 PASS
 4  2 -0.030547 -0.030547 -0.030547 -0.030547 0.090226 0.090226 0.090226 0.090226       0.0      0.0 PASS
 4  3 -0.088794 -0.088794 -0.088794 -0.088794 0.089927 0.089927 0.089927 0.089927     

### Test 47 — test_sim_attgt_matches_r[tp4_dyn-notyettreated-dr]

`tp4_dyn`, DR, notyettreated, covariates: X

In [48]:
_sim_data = load_sim('tp4_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_dyn_notyettreated_dr', st_scn='sim_tp4_dyn_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp4_dyn', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp4_dyn-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp4_dyn-notyettreated-dr] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.046131  1.046131  1.046131  1.046131 0.073752 0.073752 0.073752 0.073752       0.0      0.0 PASS
 2  3  2.072994  2.072994  2.072994  2.072994 0.076483 0.076483 0.076483 0.076483       0.0      0.0 PASS
 2  4  2.998124  2.998124  2.998124  2.998124 0.089488 0.089488 0.089488 0.089488       0.0      0.0 PASS
 3  2 -0.066215 -0.066215 -0.066215 -0.066215 0.077286 0.077286 0.077286 0.077286       0.0      0.0 PASS
 3  3  1.090567  1.090567  1.090567  1.090567 0.074064 0.074064 0.074064 0.074064       0.0      0.0 PASS
 3  4  2.121167  2.121167  2.121167  2.121167 0.089275 0.089275 0.089275 0.089275       0.0      0.0 PASS
 4  2  0.015188  0.015188  0.015188  0.015188 0.078116 0.078116 0.078116 0.078116       0.0      0.0 PASS
 4  3 -0.088217 -0.088217 -0.088217 -0.088217 0.089893 0.089893 0.089893 0.089893     

### Test 48 — test_sim_attgt_matches_r[tp4_dyn-notyettreated-reg]

`tp4_dyn`, REG, notyettreated, covariates: X

In [49]:
_sim_data = load_sim('tp4_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp4_dyn_notyettreated_reg', st_scn='sim_tp4_dyn_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp4_dyn', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp4_dyn-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp4_dyn-notyettreated-reg] (9 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.046138  1.046138  1.046138  1.046138 0.073752 0.073752 0.073752 0.073752       0.0      0.0 PASS
 2  3  2.072986  2.072986  2.072986  2.072986 0.076483 0.076483 0.076483 0.076483       0.0      0.0 PASS
 2  4  2.997542  2.997542  2.997542  2.997542 0.089460 0.089460 0.089460 0.089460       0.0      0.0 PASS
 3  2 -0.066216 -0.066216 -0.066216 -0.066216 0.077286 0.077286 0.077286 0.077286       0.0      0.0 PASS
 3  3  1.090566  1.090566  1.090566  1.090566 0.074064 0.074064 0.074064 0.074064       0.0      0.0 PASS
 3  4  2.120878  2.120878  2.120878  2.120878 0.089273 0.089273 0.089273 0.089273       0.0      0.0 PASS
 4  2  0.015597  0.015597  0.015597  0.015597 0.078121 0.078121 0.078121 0.078121       0.0      0.0 PASS
 4  3 -0.088794 -0.088794 -0.088794 -0.088794 0.089927 0.089927 0.089927 0.089927    

### Test 49 — test_sim_attgt_matches_r[tp5_dyn-nevertreated-dr]

`tp5_dyn`, DR, nevertreated, covariates: X

In [50]:
_sim_data = load_sim('tp5_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp5_dyn_nevertreated_dr', st_scn='sim_tp5_dyn_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp5_dyn', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp5_dyn-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp5_dyn-nevertreated-dr] (16 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 -0.971470 -0.971470 -0.971470 -0.971470 0.102059 0.102059 0.102059 0.102059       0.0      0.0 PASS
 2  3 -0.038862 -0.038862 -0.038862 -0.038862 0.105225 0.105225 0.105225 0.105225       0.0      0.0 PASS
 2  4  0.922646  0.922646  0.922646  0.922646 0.099250 0.099250 0.099250 0.099250       0.0      0.0 PASS
 2  5  2.030147  2.030147  2.030147  2.030147 0.103164 0.103164 0.103164 0.103164       0.0      0.0 PASS
 3  2  0.171741  0.171741  0.171741  0.171741 0.096336 0.096336 0.096336 0.096336       0.0      0.0 PASS
 3  3 -1.106290 -1.106290 -1.106290 -1.106290 0.096338 0.096338 0.096338 0.096338       0.0      0.0 PASS
 3  4 -0.169829 -0.169829 -0.169829 -0.169829 0.096686 0.096686 0.096686 0.096686       0.0      0.0 PASS
 3  5  0.769470  0.769470  0.769470  0.769470 0.093034 0.093034 0.093034 0.093034     

### Test 50 — test_sim_attgt_matches_r[tp5_dyn-nevertreated-reg]

`tp5_dyn`, REG, nevertreated, covariates: X

In [51]:
_sim_data = load_sim('tp5_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp5_dyn_nevertreated_reg', st_scn='sim_tp5_dyn_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp5_dyn', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp5_dyn-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp5_dyn-nevertreated-reg] (16 cells)


 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 -0.970612 -0.970612 -0.970612 -0.970612 0.101959 0.101959 0.101959 0.101959       0.0      0.0 PASS
 2  3 -0.038082 -0.038082 -0.038082 -0.038082 0.105160 0.105160 0.105160 0.105160       0.0      0.0 PASS
 2  4  0.924105  0.924105  0.924105  0.924105 0.099257 0.099257 0.099257 0.099257       0.0      0.0 PASS
 2  5  2.031152  2.031152  2.031152  2.031152 0.103072 0.103072 0.103072 0.103072       0.0      0.0 PASS
 3  2  0.172212  0.172212  0.172212  0.172212 0.096293 0.096293 0.096293 0.096293       0.0      0.0 PASS
 3  3 -1.106326 -1.106326 -1.106326 -1.106326 0.096398 0.096398 0.096398 0.096398       0.0      0.0 PASS
 3  4 -0.169459 -0.169459 -0.169459 -0.169459 0.096698 0.096698 0.096698 0.096698       0.0      0.0 PASS
 3  5  0.769567  0.769567  0.769567  0.769567 0.093096 0.093096 0.093096 0.093096       0.0      0.0 PASS
 4  2  0.038540  0.038540  0.038540  0.038540

### Test 51 — test_sim_attgt_matches_r[tp5_dyn-notyettreated-dr]

`tp5_dyn`, DR, notyettreated, covariates: X

In [52]:
_sim_data = load_sim('tp5_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp5_dyn_notyettreated_dr', st_scn='sim_tp5_dyn_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp5_dyn', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp5_dyn-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp5_dyn-notyettreated-dr] (16 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 -1.013610 -1.013610 -1.013610 -1.013610 0.083056 0.083056 0.083056 0.083056       0.0      0.0 PASS
 2  3 -0.044516 -0.044516 -0.044516 -0.044516 0.086253 0.086253 0.086253 0.086253       0.0      0.0 PASS
 2  4  0.894869  0.894869  0.894869  0.894869 0.086300 0.086300 0.086300 0.086300       0.0      0.0 PASS
 2  5  2.030147  2.030147  2.030147  2.030147 0.103164 0.103164 0.103164 0.103164       0.0      0.0 PASS
 3  2  0.167553  0.167553  0.167553  0.167553 0.079391 0.079391 0.079391 0.079391       0.0      0.0 PASS
 3  3 -1.107177 -1.107177 -1.107177 -1.107177 0.080745 0.080745 0.080745 0.080745       0.0      0.0 PASS
 3  4 -0.211623 -0.211623 -0.211623 -0.211623 0.085566 0.085566 0.085566 0.085566       0.0      0.0 PASS
 3  5  0.769470  0.769470  0.769470  0.769470 0.093034 0.093034 0.093034 0.093034    

### Test 52 — test_sim_attgt_matches_r[tp5_dyn-notyettreated-reg]

`tp5_dyn`, REG, notyettreated, covariates: X

In [53]:
_sim_data = load_sim('tp5_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp5_dyn_notyettreated_reg', st_scn='sim_tp5_dyn_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp5_dyn', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp5_dyn-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp5_dyn-notyettreated-reg] (16 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2 -1.013769 -1.013769 -1.013769 -1.013769 0.083067 0.083067 0.083067 0.083067       0.0      0.0 PASS
 2  3 -0.044237 -0.044237 -0.044237 -0.044237 0.086242 0.086242 0.086242 0.086242       0.0      0.0 PASS
 2  4  0.895537  0.895537  0.895537  0.895537 0.086317 0.086317 0.086317 0.086317       0.0      0.0 PASS
 2  5  2.031152  2.031152  2.031152  2.031152 0.103072 0.103072 0.103072 0.103072       0.0      0.0 PASS
 3  2  0.167460  0.167460  0.167460  0.167460 0.079395 0.079395 0.079395 0.079395       0.0      0.0 PASS
 3  3 -1.107005 -1.107005 -1.107005 -1.107005 0.080728 0.080728 0.080728 0.080728       0.0      0.0 PASS
 3  4 -0.211166 -0.211166 -0.211166 -0.211166 0.085572 0.085572 0.085572 0.085572       0.0      0.0 PASS
 3  5  0.769567  0.769567  0.769567  0.769567 0.093096 0.093096 0.093096 0.093096   

### Test 53 — test_sim_attgt_matches_r[tp8_dyn-nevertreated-dr]

`tp8_dyn`, DR, nevertreated, covariates: X

In [54]:
_sim_data = load_sim('tp8_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp8_dyn_nevertreated_dr', st_scn='sim_tp8_dyn_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp8_dyn', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp8_dyn-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp8_dyn-nevertreated-dr] (49 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.095340  1.095340  1.095340  1.095340 0.121755 0.121755 0.121755 0.121755       0.0      0.0 PASS
 2  3  1.943097  1.943097  1.943097  1.943097 0.124483 0.124483 0.124483 0.124483       0.0      0.0 PASS
 2  4  3.021794  3.021794  3.021794  3.021794 0.130197 0.130197 0.130197 0.130197       0.0      0.0 PASS
 2  5  4.112889  4.112889  4.112889  4.112889 0.129599 0.129599 0.129599 0.129599       0.0      0.0 PASS
 2  6  5.054747  5.054747  5.054747  5.054747 0.129667 0.129667 0.129667 0.129667       0.0      0.0 PASS
 2  7  6.118215  6.118215  6.118215  6.118215 0.128147 0.128147 0.128147 0.128147       0.0      0.0 PASS
 2  8  7.021447  7.021447  7.021447  7.021447 0.119281 0.119281 0.119281 0.119281       0.0      0.0 PASS
 3  2 -0.082620 -0.082620 -0.082620 -0.082620 0.130911 0.130911 0.130911 0.130911     

### Test 54 — test_sim_attgt_matches_r[tp8_dyn-nevertreated-reg]

`tp8_dyn`, REG, nevertreated, covariates: X

In [55]:
_sim_data = load_sim('tp8_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp8_dyn_nevertreated_reg', st_scn='sim_tp8_dyn_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp8_dyn', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp8_dyn-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp8_dyn-nevertreated-reg] (49 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.095288  1.095288  1.095288  1.095288 0.121752 0.121752 0.121752 0.121752       0.0      0.0 PASS
 2  3  1.943122  1.943122  1.943122  1.943122 0.124489 0.124489 0.124489 0.124489       0.0      0.0 PASS
 2  4  3.021594  3.021594  3.021594  3.021594 0.130217 0.130217 0.130217 0.130217       0.0      0.0 PASS
 2  5  4.112819  4.112819  4.112819  4.112819 0.129609 0.129609 0.129609 0.129609       0.0      0.0 PASS
 2  6  5.054816  5.054816  5.054816  5.054816 0.129661 0.129661 0.129661 0.129661       0.0      0.0 PASS
 2  7  6.118340  6.118340  6.118340  6.118340 0.128141 0.128141 0.128141 0.128141       0.0      0.0 PASS
 2  8  7.021490  7.021490  7.021490  7.021490 0.119284 0.119284 0.119284 0.119284       0.0      0.0 PASS
 3  2 -0.082621 -0.082621 -0.082621 -0.082621 0.130910 0.130910 0.130910 0.130910    

### Test 55 — test_sim_attgt_matches_r[tp8_dyn-notyettreated-dr]

`tp8_dyn`, DR, notyettreated, covariates: X

In [56]:
_sim_data = load_sim('tp8_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp8_dyn_notyettreated_dr', st_scn='sim_tp8_dyn_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp8_dyn', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp8_dyn-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp8_dyn-notyettreated-dr] (49 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.164314  1.164314  1.164314  1.164314 0.092910 0.092910 0.092910 0.092910       0.0      0.0 PASS
 2  3  2.035045  2.035045  2.035045  2.035045 0.092993 0.092993 0.092993 0.092993       0.0      0.0 PASS
 2  4  3.044593  3.044593  3.044593  3.044593 0.099358 0.099358 0.099358 0.099358       0.0      0.0 PASS
 2  5  4.079461  4.079461  4.079461  4.079461 0.101594 0.101594 0.101594 0.101594       0.0      0.0 PASS
 2  6  5.056106  5.056106  5.056106  5.056106 0.105977 0.105977 0.105977 0.105977       0.0      0.0 PASS
 2  7  6.079953  6.079953  6.079953  6.079953 0.111650 0.111650 0.111650 0.111650       0.0      0.0 PASS
 2  8  7.021447  7.021447  7.021447  7.021447 0.119281 0.119281 0.119281 0.119281       0.0      0.0 PASS
 3  2 -0.011852 -0.011852 -0.011852 -0.011852 0.105068 0.105068 0.105068 0.105068    

### Test 56 — test_sim_attgt_matches_r[tp8_dyn-notyettreated-reg]

`tp8_dyn`, REG, notyettreated, covariates: X

In [57]:
_sim_data = load_sim('tp8_dyn')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp8_dyn_notyettreated_reg', st_scn='sim_tp8_dyn_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp8_dyn', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp8_dyn-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp8_dyn-notyettreated-reg] (49 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  1.164272  1.164272  1.164272  1.164272 0.092907 0.092907 0.092907 0.092907       0.0      0.0 PASS
 2  3  2.035012  2.035012  2.035012  2.035012 0.092990 0.092990 0.092990 0.092990       0.0      0.0 PASS
 2  4  3.044570  3.044570  3.044570  3.044570 0.099358 0.099358 0.099358 0.099358       0.0      0.0 PASS
 2  5  4.079352  4.079352  4.079352  4.079352 0.101605 0.101605 0.101605 0.101605       0.0      0.0 PASS
 2  6  5.056123  5.056123  5.056123  5.056123 0.105976 0.105976 0.105976 0.105976       0.0      0.0 PASS
 2  7  6.079977  6.079977  6.079977  6.079977 0.111643 0.111643 0.111643 0.111643       0.0      0.0 PASS
 2  8  7.021490  7.021490  7.021490  7.021490 0.119284 0.119284 0.119284 0.119284       0.0      0.0 PASS
 3  2 -0.011854 -0.011854 -0.011854 -0.011854 0.105067 0.105067 0.105067 0.105067   

### Test 57 — test_sim_attgt_matches_r[tp10_const-nevertreated-dr]

`tp10_const`, DR, nevertreated, covariates: X

In [58]:
_sim_data = load_sim('tp10_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp10_const_nevertreated_dr', st_scn='sim_tp10_const_nevertreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp10_const', 'control': 'nevertreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp10_const-nevertreated-dr]', df)

PASS test_sim_attgt_matches_r[tp10_const-nevertreated-dr] (81 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.937965  0.937965  0.937965  0.937965 0.203123 0.203123 0.203123 0.203123       0.0      0.0 PASS
 2  3  0.929400  0.929400  0.929400  0.929400 0.193904 0.193904 0.193904 0.193904       0.0      0.0 PASS
 2  4  1.112329  1.112329  1.112329  1.112329 0.198642 0.198642 0.198642 0.198642       0.0      0.0 PASS
 2  5  0.872890  0.872890  0.872890  0.872890 0.205396 0.205396 0.205396 0.205396       0.0      0.0 PASS
 2  6  1.243224  1.243224  1.243224  1.243224 0.206929 0.206929 0.206929 0.206929       0.0      0.0 PASS
 2  7  1.132286  1.132286  1.132286  1.132286 0.194714 0.194714 0.194714 0.194714       0.0      0.0 PASS
 2  8  1.004829  1.004829  1.004829  1.004829 0.214290 0.214290 0.214290 0.214290       0.0      0.0 PASS
 2  9  1.169981  1.169981  1.169981  1.169981 0.221230 0.221230 0.221230 0.221230  

### Test 58 — test_sim_attgt_matches_r[tp10_const-nevertreated-reg]

`tp10_const`, REG, nevertreated, covariates: X

In [59]:
_sim_data = load_sim('tp10_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='nevertreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp10_const_nevertreated_reg', st_scn='sim_tp10_const_nevertreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp10_const', 'control': 'nevertreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp10_const-nevertreated-reg]', df)

PASS test_sim_attgt_matches_r[tp10_const-nevertreated-reg] (81 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.937953  0.937953  0.937953  0.937953 0.203138 0.203138 0.203138 0.203138       0.0      0.0 PASS
 2  3  0.929382  0.929382  0.929382  0.929382 0.193919 0.193919 0.193919 0.193919       0.0      0.0 PASS
 2  4  1.112314  1.112314  1.112314  1.112314 0.198645 0.198645 0.198645 0.198645       0.0      0.0 PASS
 2  5  0.872885  0.872885  0.872885  0.872885 0.205408 0.205408 0.205408 0.205408       0.0      0.0 PASS
 2  6  1.243221  1.243221  1.243221  1.243221 0.206925 0.206925 0.206925 0.206925       0.0      0.0 PASS
 2  7  1.132237  1.132237  1.132237  1.132237 0.194761 0.194761 0.194761 0.194761       0.0      0.0 PASS
 2  8  1.004785  1.004785  1.004785  1.004785 0.214339 0.214339 0.214339 0.214339       0.0      0.0 PASS
 2  9  1.169953  1.169953  1.169953  1.169953 0.221223 0.221223 0.221223 0.221223 

### Test 59 — test_sim_attgt_matches_r[tp10_const-notyettreated-dr]

`tp10_const`, DR, notyettreated, covariates: X

In [60]:
_sim_data = load_sim('tp10_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='dr', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp10_const_notyettreated_dr', st_scn='sim_tp10_const_notyettreated_dr',
                r_df=r_sim, r_filt={'dataset': 'tp10_const', 'control': 'notyettreated', 'est': 'dr'})
compare_test('test_sim_attgt_matches_r[tp10_const-notyettreated-dr]', df)

PASS test_sim_attgt_matches_r[tp10_const-notyettreated-dr] (81 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.823894  0.823894  0.823894  0.823894 0.155406 0.155406 0.155406 0.155406       0.0      0.0 PASS
 2  3  0.845708  0.845708  0.845708  0.845708 0.154904 0.154904 0.154904 0.154904       0.0      0.0 PASS
 2  4  1.034315  1.034315  1.034315  1.034315 0.149153 0.149153 0.149153 0.149153       0.0      0.0 PASS
 2  5  0.962331  0.962331  0.962331  0.962331 0.148591 0.148591 0.148591 0.148591       0.0      0.0 PASS
 2  6  1.062685  1.062685  1.062685  1.062685 0.161748 0.161748 0.161748 0.161748       0.0      0.0 PASS
 2  7  1.006236  1.006236  1.006236  1.006236 0.153519 0.153519 0.153519 0.153519       0.0      0.0 PASS
 2  8  0.968451  0.968451  0.968451  0.968451 0.166254 0.166254 0.166254 0.166254       0.0      0.0 PASS
 2  9  1.137783  1.137783  1.137783  1.137783 0.192789 0.192789 0.192789 0.192789 

### Test 60 — test_sim_attgt_matches_r[tp10_const-notyettreated-reg]

`tp10_const`, REG, notyettreated, covariates: X

In [61]:
_sim_data = load_sim('tp10_const')
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(data=_sim_data, yname='Y', tname='period', idname='id',
            gname='G', xformla='Y~X', control_group='notyettreated'
            ).fit(est_method='reg', bstrap=False)
df = prep_attgt(res, jl_scn='sim_tp10_const_notyettreated_reg', st_scn='sim_tp10_const_notyettreated_reg',
                r_df=r_sim, r_filt={'dataset': 'tp10_const', 'control': 'notyettreated', 'est': 'reg'})
compare_test('test_sim_attgt_matches_r[tp10_const-notyettreated-reg]', df)

PASS test_sim_attgt_matches_r[tp10_const-notyettreated-reg] (81 cells)
 g  t     R_att    Py_att    Jl_att    St_att     R_se    Py_se    Jl_se    St_se  att_diff  se_diff   ok
 2  2  0.823913  0.823913  0.823913  0.823913 0.155385 0.155385 0.155385 0.155385       0.0      0.0 PASS
 2  3  0.845703  0.845703  0.845703  0.845703 0.154918 0.154918 0.154918 0.154918       0.0      0.0 PASS
 2  4  1.034257  1.034257  1.034257  1.034257 0.149161 0.149161 0.149161 0.149161       0.0      0.0 PASS
 2  5  0.962330  0.962330  0.962330  0.962330 0.148583 0.148583 0.148583 0.148583       0.0      0.0 PASS
 2  6  1.062676  1.062676  1.062676  1.062676 0.161739 0.161739 0.161739 0.161739       0.0      0.0 PASS
 2  7  1.006188  1.006188  1.006188  1.006188 0.153512 0.153512 0.153512 0.153512       0.0      0.0 PASS
 2  8  0.968429  0.968429  0.968429  0.968429 0.166283 0.166283 0.166283 0.166283       0.0      0.0 PASS
 2  9  1.137646  1.137646  1.137646  1.137646 0.192771 0.192771 0.192771 0.192771

## H. test_jel_replication: JEL article replication (5 tests)

These tests replicate results from the Callaway & Sant'Anna JEL article using
`county_mortality_data.csv`. R reference values are the published article values.
Julia references from `julia_jel_results.csv` and `julia_jel_aggte.csv`.
No Stata data for JEL tests (complex data preparation not in wrapper).

### Test 61 — test_jel_table7_2x2

JEL Table 7: 2x2 CS-DiD, 6 method/weight combos against published values

In [62]:
jel_path = os.path.join(PROJECT, 'data', 'county_mortality_data.csv')
mydata = pd.read_csv(jel_path, dtype={'county': str})
mydata['state'] = mydata['county'].str[-2:]
mydata = mydata[~mydata['state'].isin(['DC','DE','MA','NY','VT'])].copy()
mydata = mydata[(mydata['yaca'] == 2014) | mydata['yaca'].isna() | (mydata['yaca'] > 2019)].copy()
mydata['perc_white'] = mydata['population_20_64_white'] / mydata['population_20_64'] * 100
mydata['perc_hispanic'] = mydata['population_20_64_hispanic'] / mydata['population_20_64'] * 100
mydata['perc_female'] = mydata['population_20_64_female'] / mydata['population_20_64'] * 100
mydata['unemp_rate'] = mydata['unemp_rate'] * 100
mydata['median_income'] = mydata['median_income'] / 1000
keep = ['county_code','year','population_20_64','yaca','crude_rate_20_64',
        'perc_female','perc_white','perc_hispanic','unemp_rate','poverty_rate','median_income']
mydata = mydata[keep].dropna(subset=[c for c in keep if c != 'yaca'])
both = mydata[mydata['year'].isin([2013,2014])].groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(both[both==2].index)].copy()
full = mydata.groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(full[full==11].index)].copy()
short = mydata[mydata['year'].isin([2013,2014])].copy()
short['treat_year'] = short['yaca'].apply(lambda x: 2014 if pd.notna(x) and x == 2014 else 0)
short['county_code'] = short['county_code'].astype(float)
wt = short.loc[short['year']==2013, ['county_code','population_20_64']].copy()
wt.columns = ['county_code','set_wt']
short = short.merge(wt, on='county_code')

COVS = 'crude_rate_20_64 ~ perc_female + perc_white + perc_hispanic + unemp_rate + poverty_rate + median_income'
expected = {
    ('reg', None): -1.6154372119, ('ipw', None): -0.8585625501, ('dr', None): -1.2256473242,
    ('reg', 'set_wt'): -3.4592200594, ('ipw', 'set_wt'): -3.8416966846, ('dr', 'set_wt'): -3.7561045985,
}

rows = []
for (method, wt_name), exp_att in expected.items():
    with contextlib.redirect_stdout(io.StringIO()):
        res = ATTgt(yname='crude_rate_20_64', tname='year', idname='county_code',
                    gname='treat_year', xformla=COVS, data=short,
                    control_group='nevertreated', weights_name=wt_name
                    ).fit(est_method=method, base_period='universal', bstrap=False)
        res.aggte(typec='simple', na_rm=True, bstrap=False)
    py_att = float(res.atte['overall_att'])
    wlabel = 'wt' if wt_name else 'unwt'
    jl_scn = f'table7_{method}_{wlabel}'
    jl_sub = jl_jel_agg[(jl_jel_agg['scenario']==jl_scn) & (jl_jel_agg['agg_type']=='simple')]
    jl_att = float(jl_sub.iloc[0]['overall_att']) if not jl_sub.empty else np.nan
    rows.append({'g': f'{method}/{wlabel}', 't': '-',
                 'R_att': exp_att, 'Py_att': py_att, 'Jl_att': jl_att, 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

compare_test('test_jel_table7_2x2', pd.DataFrame(rows))

PASS test_jel_table7_2x2 (6 cells)
       g t     R_att    Py_att    Jl_att  St_att  R_se  Py_se  Jl_se  St_se  att_diff  se_diff   ok
reg/unwt - -1.615437 -1.615437 -1.615437     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
ipw/unwt - -0.858563 -0.858563 -0.858563     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
 dr/unwt - -1.225647 -1.225647 -1.225647     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
  reg/wt - -3.459220 -3.459220 -3.459220     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
  ipw/wt - -3.841697 -3.841697 -3.841697     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   dr/wt - -3.756105 -3.756105 -3.756105     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS


### Test 62 — test_jel_2xt_event_study

JEL 2xT: event study ATT(g,t) + dynamic aggregation, weighted REG

In [63]:
jel_path = os.path.join(PROJECT, 'data', 'county_mortality_data.csv')
mydata = pd.read_csv(jel_path, dtype={'county': str})
mydata['state'] = mydata['county'].str[-2:]
mydata = mydata[~mydata['state'].isin(['DC','DE','MA','NY','VT'])].copy()
mydata = mydata[(mydata['yaca'] == 2014) | mydata['yaca'].isna() | (mydata['yaca'] > 2019)].copy()
mydata['perc_white'] = mydata['population_20_64_white'] / mydata['population_20_64'] * 100
mydata['perc_hispanic'] = mydata['population_20_64_hispanic'] / mydata['population_20_64'] * 100
mydata['perc_female'] = mydata['population_20_64_female'] / mydata['population_20_64'] * 100
mydata['unemp_rate'] = mydata['unemp_rate'] * 100
mydata['median_income'] = mydata['median_income'] / 1000
keep = ['county_code','year','population_20_64','yaca','crude_rate_20_64',
        'perc_female','perc_white','perc_hispanic','unemp_rate','poverty_rate','median_income']
mydata = mydata[keep].dropna(subset=[c for c in keep if c != 'yaca'])
both = mydata[mydata['year'].isin([2013,2014])].groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(both[both==2].index)].copy()
full = mydata.groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(full[full==11].index)].copy()
mydata['treat_year'] = mydata['yaca'].apply(lambda x: 2014 if pd.notna(x) and x == 2014 else 0)
mydata['county_code'] = mydata['county_code'].astype(float)
wt = mydata.loc[mydata['year']==2013, ['county_code','population_20_64']].copy()
wt.columns = ['county_code','set_wt']
mydata = mydata.merge(wt, on='county_code')

with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(yname='crude_rate_20_64', tname='year', idname='county_code',
                gname='treat_year', data=mydata, weights_name='set_wt',
                control_group='nevertreated'
                ).fit(est_method='reg', base_period='universal', bstrap=False)

expected_att = [4.1292044043, -0.5016807242, 2.7531791360, 2.7804626426,
                0.0, -2.5628745138, -1.6973291127, 0.2189009815,
                -0.8133358354, -1.1532954495, 1.7866564429]

py = _extract_py(res)
jl = _ref_lookup(jl_jel, {'scenario': '2xt_event'})
r_map = dict(zip(sorted(py.keys()), [(e, np.nan) for e in expected_att]))
rows = []
for k in sorted(py.keys()):
    rows.append({'g': k[0], 't': k[1],
                 'R_att': r_map.get(k, (np.nan,))[0], 'Py_att': py[k][0],
                 'Jl_att': jl.get(k, (np.nan,np.nan))[0], 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': py[k][1],
                 'Jl_se': jl.get(k, (np.nan,np.nan))[1], 'St_se': np.nan})

with contextlib.redirect_stdout(io.StringIO()):
    res.aggte(typec='dynamic', min_e=0, max_e=5, bstrap=False)
py_ov = float(res.atte['overall_att'])
jl_ov_sub = jl_jel_agg[(jl_jel_agg['scenario']=='2xt_event') & (jl_jel_agg['agg_type']=='dynamic')]
jl_ov = float(jl_ov_sub.iloc[0]['overall_att']) if not jl_ov_sub.empty else np.nan
rows.append({'g': 'ov(0-5)', 't': '-',
             'R_att': -0.7035462478, 'Py_att': py_ov, 'Jl_att': jl_ov, 'St_att': np.nan,
             'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

compare_test('test_jel_2xt_event_study', pd.DataFrame(rows))

PASS test_jel_2xt_event_study (12 cells)
      g    t     R_att    Py_att    Jl_att  St_att  R_se    Py_se    Jl_se  St_se  att_diff  se_diff   ok
   2014 2009  4.129204  4.129204  4.129204     NaN   NaN 2.631169 2.631169    NaN       0.0      0.0 PASS
   2014 2010 -0.501681 -0.501681 -0.501681     NaN   NaN 2.026858 2.026858    NaN       0.0      0.0 PASS
   2014 2011  2.753179  2.753179  2.753179     NaN   NaN 1.615970 1.615970    NaN       0.0      0.0 PASS
   2014 2012  2.780463  2.780463  2.780463     NaN   NaN 1.522273 1.522273    NaN       0.0      0.0 PASS
   2014 2013  0.000000  0.000000  0.000000     NaN   NaN      NaN      NaN    NaN       0.0      NaN PASS
   2014 2014 -2.562874 -2.562874 -2.562874     NaN   NaN 1.489160 1.489160    NaN       0.0      0.0 PASS
   2014 2015 -1.697329 -1.697329 -1.697329     NaN   NaN 1.838074 1.838074    NaN       0.0      0.0 PASS
   2014 2016  0.218901  0.218901  0.218901     NaN   NaN 2.330400 2.330400    NaN       0.0      0.0 PASS
   20

### Test 63 — test_jel_2xt_with_covariates

JEL 2xT with covariates: e=-1 should be 0, all estimates finite (REG/IPW/DR)

In [64]:
jel_path = os.path.join(PROJECT, 'data', 'county_mortality_data.csv')
mydata = pd.read_csv(jel_path, dtype={'county': str})
mydata['state'] = mydata['county'].str[-2:]
mydata = mydata[~mydata['state'].isin(['DC','DE','MA','NY','VT'])].copy()
mydata = mydata[(mydata['yaca'] == 2014) | mydata['yaca'].isna() | (mydata['yaca'] > 2019)].copy()
mydata['perc_white'] = mydata['population_20_64_white'] / mydata['population_20_64'] * 100
mydata['perc_hispanic'] = mydata['population_20_64_hispanic'] / mydata['population_20_64'] * 100
mydata['perc_female'] = mydata['population_20_64_female'] / mydata['population_20_64'] * 100
mydata['unemp_rate'] = mydata['unemp_rate'] * 100
mydata['median_income'] = mydata['median_income'] / 1000
keep = ['county_code','year','population_20_64','yaca','crude_rate_20_64',
        'perc_female','perc_white','perc_hispanic','unemp_rate','poverty_rate','median_income']
mydata = mydata[keep].dropna(subset=[c for c in keep if c != 'yaca'])
both = mydata[mydata['year'].isin([2013,2014])].groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(both[both==2].index)].copy()
full = mydata.groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(full[full==11].index)].copy()
mydata['treat_year'] = mydata['yaca'].apply(lambda x: 2014 if pd.notna(x) and x == 2014 else 0)
mydata['county_code'] = mydata['county_code'].astype(float)
wt = mydata.loc[mydata['year']==2013, ['county_code','population_20_64']].copy()
wt.columns = ['county_code','set_wt']
mydata = mydata.merge(wt, on='county_code')

COVS = 'crude_rate_20_64 ~ perc_female + perc_white + perc_hispanic + unemp_rate + poverty_rate + median_income'
rows = []
for method in ['reg', 'ipw', 'dr']:
    with contextlib.redirect_stdout(io.StringIO()):
        res = ATTgt(yname='crude_rate_20_64', tname='year', idname='county_code',
                    gname='treat_year', xformla=COVS, data=mydata,
                    weights_name='set_wt', control_group='nevertreated'
                    ).fit(est_method=method, base_period='universal', bstrap=False)
        res.aggte(typec='dynamic', na_rm=True, bstrap=False)
    egt = np.asarray(res.atte['egt'], dtype=float)
    att_egt = np.asarray(res.atte['att_egt'], dtype=float)
    base_idx = np.where(egt == -1)[0]
    base_val = float(att_egt[base_idx[0]]) if len(base_idx) > 0 else np.nan
    all_finite = bool(np.all(np.isfinite(att_egt[~np.isnan(att_egt)])))
    jl_scn = f'2xt_cov_{method}'
    jl_sub = jl_jel_agg[(jl_jel_agg['scenario']==jl_scn) & (jl_jel_agg['agg_type']=='dynamic')]
    jl_egt_sub = jl_sub[jl_sub['egt'].notna() & (jl_sub['egt'] == -1.0)]
    jl_base = float(jl_egt_sub.iloc[0]['att_egt']) if not jl_egt_sub.empty else np.nan
    rows.append({'g': f'{method} e=-1', 't': '-',
                 'R_att': 0.0, 'Py_att': base_val, 'Jl_att': jl_base, 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})
    rows.append({'g': f'{method} finite', 't': '-',
                 'R_att': 1.0, 'Py_att': 1.0 if all_finite else 0.0, 'Jl_att': 1.0, 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

compare_test('test_jel_2xt_with_covariates', pd.DataFrame(rows))

PASS test_jel_2xt_with_covariates (6 cells)
         g t  R_att  Py_att  Jl_att  St_att  R_se  Py_se  Jl_se  St_se  att_diff  se_diff   ok
  reg e=-1 -    0.0     0.0     0.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
reg finite -    1.0     1.0     1.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
  ipw e=-1 -    0.0     0.0     0.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
ipw finite -    1.0     1.0     1.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   dr e=-1 -    0.0     0.0     0.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
 dr finite -    1.0     1.0     1.0     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS


### Test 64 — test_jel_gxt_no_covs

JEL GxT: staggered event study, no covariates, notyettreated, weighted REG

In [65]:
jel_path = os.path.join(PROJECT, 'data', 'county_mortality_data.csv')
mydata = pd.read_csv(jel_path, dtype={'county': str})
mydata['state'] = mydata['county'].str[-2:]
mydata = mydata[~mydata['state'].isin(['DC','DE','MA','NY','VT'])].copy()
mydata['perc_white'] = mydata['population_20_64_white'] / mydata['population_20_64'] * 100
mydata['perc_hispanic'] = mydata['population_20_64_hispanic'] / mydata['population_20_64'] * 100
mydata['perc_female'] = mydata['population_20_64_female'] / mydata['population_20_64'] * 100
mydata['unemp_rate'] = mydata['unemp_rate'] * 100
mydata['median_income'] = mydata['median_income'] / 1000
keep = ['county_code','year','population_20_64','yaca','crude_rate_20_64',
        'perc_female','perc_white','perc_hispanic','unemp_rate','poverty_rate','median_income']
mydata = mydata[keep].dropna(subset=[c for c in keep if c != 'yaca'])
both = mydata[mydata['year'].isin([2013,2014])].groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(both[both==2].index)].copy()
full = mydata.groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(full[full==11].index)].copy()
mydata['treat_year'] = mydata['yaca'].apply(lambda x: int(x) if pd.notna(x) and x <= 2019 else 0)
mydata['county_code'] = mydata['county_code'].astype(float)
wt = mydata.loc[mydata['year']==2013, ['county_code','population_20_64']].copy()
wt.columns = ['county_code','set_wt']
mydata = mydata.merge(wt, on='county_code')

with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(yname='crude_rate_20_64', tname='year', idname='county_code',
                gname='treat_year', data=mydata, weights_name='set_wt',
                control_group='notyettreated'
                ).fit(est_method='reg', base_period='universal', bstrap=False)
    res.aggte(typec='dynamic', bstrap=False)

expected = {-5: 2.2186783578, -4: 0.8579225109, -3: 1.9161499399,
            -2: 2.5644742860, -1: 0.0,
            0: -1.6545648988, 1: -0.2616435324, 2: 1.7055625922,
            3: -0.5405232028, 4: -0.5148819184, 5: 1.7866564429}

egt = np.asarray(res.atte['egt'], dtype=float)
att_egt = np.asarray(res.atte['att_egt'], dtype=float)
py_map = {float(e): a for e, a in zip(egt, att_egt)}

jl_map_raw, jl_ov = _ref_aggte_egt(jl_jel_agg, {'scenario': 'gxt_nocov'}, 'dynamic', type_col='agg_type')

rows = []
for e_val in sorted(expected.keys()):
    rows.append({'g': f'e={e_val}', 't': '-',
                 'R_att': expected[e_val], 'Py_att': py_map.get(e_val, np.nan),
                 'Jl_att': jl_map_raw.get(float(e_val), (np.nan,np.nan))[0], 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

with contextlib.redirect_stdout(io.StringIO()):
    res.aggte(typec='dynamic', min_e=0, max_e=5, bstrap=False)
py_ov = float(res.atte['overall_att'])
jl_ov_sub = jl_jel_agg[(jl_jel_agg['scenario']=='gxt_nocov_05') & (jl_jel_agg['agg_type']=='dynamic')]
jl_ov_val = float(jl_ov_sub.iloc[0]['overall_att']) if not jl_ov_sub.empty else np.nan
rows.append({'g': 'ov(0-5)', 't': '-',
             'R_att': 0.0867675805, 'Py_att': py_ov, 'Jl_att': jl_ov_val, 'St_att': np.nan,
             'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

compare_test('test_jel_gxt_no_covs', pd.DataFrame(rows))

PASS test_jel_gxt_no_covs (12 cells)
      g t     R_att    Py_att    Jl_att  St_att  R_se  Py_se  Jl_se  St_se  att_diff  se_diff   ok
   e=-5 -  2.218678  2.218678  2.218678     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-4 -  0.857923  0.857923  0.857923     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-3 -  1.916150  1.916150  1.916150     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-2 -  2.564474  2.564474  2.564474     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-1 -  0.000000  0.000000  0.000000     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=0 - -1.654565 -1.654565 -1.654565     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=1 - -0.261643 -0.261643 -0.261643     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=2 -  1.705563  1.705563  1.705563     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=3 - -0.540523 -0.540523 -0.540523     NaN   NaN    NaN    NaN    N

### Test 65 — test_jel_gxt_dr_covs

JEL GxT: staggered DR with covariates, notyettreated, weighted

In [66]:
jel_path = os.path.join(PROJECT, 'data', 'county_mortality_data.csv')
mydata = pd.read_csv(jel_path, dtype={'county': str})
mydata['state'] = mydata['county'].str[-2:]
mydata = mydata[~mydata['state'].isin(['DC','DE','MA','NY','VT'])].copy()
mydata['perc_white'] = mydata['population_20_64_white'] / mydata['population_20_64'] * 100
mydata['perc_hispanic'] = mydata['population_20_64_hispanic'] / mydata['population_20_64'] * 100
mydata['perc_female'] = mydata['population_20_64_female'] / mydata['population_20_64'] * 100
mydata['unemp_rate'] = mydata['unemp_rate'] * 100
mydata['median_income'] = mydata['median_income'] / 1000
keep = ['county_code','year','population_20_64','yaca','crude_rate_20_64',
        'perc_female','perc_white','perc_hispanic','unemp_rate','poverty_rate','median_income']
mydata = mydata[keep].dropna(subset=[c for c in keep if c != 'yaca'])
both = mydata[mydata['year'].isin([2013,2014])].groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(both[both==2].index)].copy()
full = mydata.groupby('county_code').size()
mydata = mydata[mydata['county_code'].isin(full[full==11].index)].copy()
mydata['treat_year'] = mydata['yaca'].apply(lambda x: int(x) if pd.notna(x) and x <= 2019 else 0)
mydata['county_code'] = mydata['county_code'].astype(float)
wt = mydata.loc[mydata['year']==2013, ['county_code','population_20_64']].copy()
wt.columns = ['county_code','set_wt']
mydata = mydata.merge(wt, on='county_code')

COVS = 'crude_rate_20_64 ~ perc_female + perc_white + perc_hispanic + unemp_rate + poverty_rate + median_income'
with contextlib.redirect_stdout(io.StringIO()):
    res = ATTgt(yname='crude_rate_20_64', tname='year', idname='county_code',
                gname='treat_year', xformla=COVS, data=mydata,
                weights_name='set_wt', control_group='notyettreated'
                ).fit(est_method='dr', base_period='universal', bstrap=False)

expected = {-5: 2.6684811691, -4: 2.1333836537, -3: 2.8574389179,
            -2: 2.9129673584, -1: 0.0,
            0: -1.4929553032, 1: -1.9693922262, 2: -2.7250659699,
            3: -5.0556625460, 4: -4.7161630370, 5: 2.4772492894}

with contextlib.redirect_stdout(io.StringIO()):
    res.aggte(typec='dynamic', bstrap=False)
egt = np.asarray(res.atte['egt'], dtype=float)
att_egt = np.asarray(res.atte['att_egt'], dtype=float)
py_map = {float(e): a for e, a in zip(egt, att_egt)}

jl_map_raw, jl_ov = _ref_aggte_egt(jl_jel_agg, {'scenario': 'gxt_dr_cov'}, 'dynamic', type_col='agg_type')

rows = []
for e_val in sorted(expected.keys()):
    rows.append({'g': f'e={e_val}', 't': '-',
                 'R_att': expected[e_val], 'Py_att': py_map.get(e_val, np.nan),
                 'Jl_att': jl_map_raw.get(float(e_val), (np.nan,np.nan))[0], 'St_att': np.nan,
                 'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

with contextlib.redirect_stdout(io.StringIO()):
    res.aggte(typec='dynamic', min_e=0, max_e=5, bstrap=False)
py_ov = float(res.atte['overall_att'])
jl_ov_sub = jl_jel_agg[(jl_jel_agg['scenario']=='gxt_dr_cov_05') & (jl_jel_agg['agg_type']=='dynamic')]
jl_ov_val = float(jl_ov_sub.iloc[0]['overall_att']) if not jl_ov_sub.empty else np.nan
rows.append({'g': 'ov(0-5)', 't': '-',
             'R_att': -2.2469982988, 'Py_att': py_ov, 'Jl_att': jl_ov_val, 'St_att': np.nan,
             'R_se': np.nan, 'Py_se': np.nan, 'Jl_se': np.nan, 'St_se': np.nan})

compare_test('test_jel_gxt_dr_covs', pd.DataFrame(rows))

PASS test_jel_gxt_dr_covs (12 cells)
      g t     R_att    Py_att    Jl_att  St_att  R_se  Py_se  Jl_se  St_se  att_diff  se_diff   ok
   e=-5 -  2.668481  2.668481  2.668481     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-4 -  2.133384  2.133384  2.133384     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-3 -  2.857439  2.857439  2.857439     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-2 -  2.912967  2.912967  2.912967     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
   e=-1 -  0.000000  0.000000  0.000000     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=0 - -1.492955 -1.492955 -1.492955     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=1 - -1.969392 -1.969392 -1.969392     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=2 - -2.725066 -2.725066 -2.725066     NaN   NaN    NaN    NaN    NaN       0.0      NaN PASS
    e=3 - -5.055663 -5.055663 -5.055663     NaN   NaN    NaN    NaN    N

## Global Summary

In [67]:
n_pass = sum(1 for r in results_tracker if r['status'] == 'PASS')
n_fail = sum(1 for r in results_tracker if r['status'] == 'FAIL')
n_total = len(results_tracker)

summary_df = pd.DataFrame(results_tracker)
summary_df.index = range(1, len(summary_df) + 1)
summary_df.index.name = 'Test #'
print(summary_df.to_string())

print()
if n_fail == 0:
    print(f"Result: {n_pass}/{n_total} tests PASS")
else:
    print(f"Result: {n_pass}/{n_total} PASS, {n_fail} FAIL")
    print("\nFailed tests:")
    for r in results_tracker:
        if r['status'] == 'FAIL':
            print(f"  x {r['name']}")

                                                          name status
Test #                                                               
1                           test_attgt_matches_r[mpdta_nev_dr]   PASS
2                           test_attgt_matches_r[mpdta_nyt_dr]   PASS
3                      test_attgt_matches_r[mpdta_nev_reg_cov]   PASS
4                          test_attgt_matches_r[mpdta_nev_ipw]   PASS
5                             test_attgt_matches_r[sim_nev_dr]   PASS
6                   test_aggte_overall_matches_r[mpdta_nev_dr]   PASS
7                   test_aggte_overall_matches_r[mpdta_nyt_dr]   PASS
8              test_aggte_overall_matches_r[mpdta_nev_reg_cov]   PASS
9                  test_aggte_overall_matches_r[mpdta_nev_ipw]   PASS
10                    test_aggte_overall_matches_r[sim_nev_dr]   PASS
11                test_aggte_egt_matches_r[group-mpdta_nev_dr]   PASS
12                test_aggte_egt_matches_r[group-mpdta_nyt_dr]   PASS
13           test_ag